# EDA/SDA 2 (Lean): Scope Cleaning and Behavior

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

`EDA_SDA_1.ipynb` cleaned the full file, compared every field and cut the scope: **the roles TripleTen trains for, plus adjacent tech roles** (about 0.9M listings). This notebook does the **non-text** in-depth analysis of that scope: additional cleaning (staffing agencies, contract types, titles, the app's filter fields), ghost listings vs. employers still hiring, source, freshness, pay, reposts and copies, company facts, missing data, outliers, what a career changer can reach, and the **retraining compass's 2026 side** by role bucket and TripleTen program. It saves **its own final table** for `EDA_SDA_3.ipynb` (text), then `EDA_SDA_4.ipynb` (statistics), `TimeSeries`, `ML_Fit` and `ML_Rank`.

**Inputs from EDA_SDA_1:** the in-scope table (`analysis_table.parquet`: English, listings that name the US, the scope filter applied; every candidate factor; role buckets; `tripleten_program`; `company_earlier` in any field; crawl-artifact flags), and company activity in any field (`company_activity.parquet`).

**Conventions:**
- Every decision is recorded with its reasoning; fix or flag, never drop.
- Every factor is checked against the confounders (section 2): within groups, not by filtering.
- Aggregates only (shares, medians, compact counts); never rows, except the cell tagged `data-rows` (the 2b.1 review of flagged company names): clear it before committing.
- Each table or chart group ends with **What the data shows**: up to three bullets.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Product |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, and what's the scope? | The scope table |
| **EDA_SDA_2** | **Which scope listings do we keep, and how do they behave?** | **Cleaned and flagged scope table (ghosts, hiding rules)** |
| EDA_SDA_3 | What do the descriptions say? | Text features and O*NET skills |
| EDA_SDA_4 | Which features hold up? | `modeling_ready.parquet` (the features that survived) |
| TimeSeries | When do listings appear and close? | Timing patterns by month |
| Profiles | Who really changed into or within tech or math, and how? (frozen) | The 2,813 career changers |
| ML_Fit | Which roles does a career changer fit? | The trained layer (role fit), saved for the app |
| ML_Rank | Which listings should they apply to first? | Listing scores and the queue's weights, saved for the app |
| Evaluate | How well do the app's rankers do on real career moves? | The comparison: `rrf`, `mlrank`, combined |

### Clarifying questions
- Which long-open listings are ghost listings, and which are employers still trying to hire?
- Does source matter beyond company hiring volume, or is it company volume in disguise?
- How precise is freshness, given the PredictLeads crawler finds listings a median of 1.5 days late?
- Is pay disclosure driven by state pay-transparency laws or job type, rather than employer intent?
- Does high company volume mean more competition, or more openings to win? And are fast-closing recruiters just staffing agencies?
- How are values missing: at random, or tied to groups (source, role bucket, state)?
- How much of the scope is left once the reliable hiding rules (seniority, likely ghosts, non-tech occupations) are applied, and how much more would an opt-in text filter hide?
- Which role buckets have enough listings and enough career changers for their own analysis?

### Roadmap
1. **Setup:** EDA_SDA_1's in-scope table and company activity
2. **Analysis views and confounders;** **additional cleaning** (2b): staffing agencies and recruiters, dates, contract types and job type, titles (with an AI/ML flag), multi-location, the app's filter fields
3. **Checks and analyses** (non-text): the compass's 2026 side (3.12) and the **feature screening** (3.13)
4. **EDA_SDA_2 final table:** decisions defended, then applied, with a reason for each step
5. **Conclusion and next steps** (into EDA_SDA_3)

### About this Lean draft

A plain-pandas rewrite of `EDA_SDA_2`, built the way `EDA_SDA_1_Lean` was: no shared formatting or summary-table helpers (`src/eda_helpers.py`), every table printed with `display()` and an explicit format, the Kaplan-Meier method from `src/kaplan_meier.py`. The standing rules are in `docs/lean_conventions.md`. This notebook reads `EDA_SDA_1_Lean`'s tables in `data/clean/lean/` (so running it is also the check that `EDA_SDA_1` can be retired, `../ANALYSIS.md`) and writes its own tables there too, not beside `EDA_SDA_2`'s, until it replaces the original. The Purpose, clarifying questions and roadmap above are `EDA_SDA_2`'s, unchanged.

Covers sections 1, 2 and 2b so far; see the closing note at the end for what's still open.

## 1. Setup

In [ ]:
# 1a Libraries and the project's own code (no shared formatting helpers)
import gc
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

# Add analysis/ to Python's search path, so "from src..." imports work from inside notebooks/
PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "pl_data.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))
from src import kaplan_meier as km  # the survival method (docs/kaplan_meier.md)
from src.career_switch import (EXTRA_ORDER, MARKETING_BUCKET, PROGRAM_ORDER, ROLE_ORDER, RULES, marketing_specialist,
                               role_bucket)  # role buckets; the senior title rule
from src.pl_data import DATA_DIR, map_parts  # the companies table, loaded one saved part at a time
from src.rows_table import scope_values  # one original field for in-scope listings, from the stream's saved records

pd.set_option("display.max_rows", 200)   # long summary tables show in full
pd.set_option("display.max_colwidth", 140)

CLEAN_DIR = DATA_DIR / "clean"
LEAN_DIR = CLEAN_DIR / "lean"  # EDA_SDA_1_Lean's tables are read from here, and this notebook writes its own here too

In [ ]:
# 1a-2 The chart style every chart below uses (src/eda_helpers.py's use_chart_style(), written out here). The four
# categorical colours are in a fixed order that is never reordered, so a given group keeps the same colour in every chart
# and every notebook; ORANGE is reserved for the one group being highlighted.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#c9c8c2", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "font.size": 10.5, "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})


def bar_h(ax, values, title, xlabel, fmt="{:,.0f}", color=BLUE):
    """Horizontal bars, largest at the top, each value written at the end of its bar."""
    values = values.dropna().sort_values()
    ax.barh([str(label)[:42] for label in values.index], values.values, color=color, height=0.65)
    for row, value in enumerate(values.values):
        ax.text(value, row, " " + fmt.format(value), va="center", color=INK_2, fontsize=8.5)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.18)


def as_list(v):
    """Nested list fields come back as lists or numpy arrays; normalize to a list."""
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    return [] if v is None or (isinstance(v, float) and np.isnan(v)) else [v]

In [ ]:
# 1b Load EDA_SDA_1_Lean's in-scope table (its section 7): the roles TripleTen trains for, plus adjacent tech roles.
# Columns this notebook doesn't use stay in that table (memory: 1.0M listings in 8 GB).
SCOPE_PATH = LEAN_DIR / "analysis_table.parquet"
if not SCOPE_PATH.exists():
    raise FileNotFoundError("Run EDA_SDA_1_Lean first: its section 7 saves the in-scope table.")
SKIP = ["tags", "categories", "last_processed", "salary_text", "location_text", "job_title", "job_title_fallback", "row",
        "record_type", "company_record_type", "was_translated"]
part1 = pd.read_parquet(SCOPE_PATH, columns=[c for c in pq.read_schema(SCOPE_PATH).names if c not in SKIP])
for c in part1.columns:  # true / false / unknown flags come back from Parquet as objects: make them nullable booleans
    if part1[c].dtype == object:
        sample = part1[c].dropna().head(1000)
        if len(sample) and sample.map(lambda v: isinstance(v, (bool, np.bool_))).all():
            part1[c] = part1[c].astype("boolean")
print(f"EDA_SDA_1_Lean in-scope table: {len(part1):,} listings x {part1.shape[1]} columns, "
      f"{part1.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

In [ ]:
# 1c From EDA_SDA_1_Lean's section 3 (every field, before the cut): company activity, for the "company still posting"
# ghost signal (3.1). The occupation summary (pay and timing per occupation) is also saved there, for the compass.
company_activity = pd.read_parquet(LEAN_DIR / "company_activity.parquet")
company_activity.index = company_activity.index.astype(str)
company_last_new = company_activity.newest_listing_any_field  # newest listing in any field, one date per company
print(f"company activity: {len(company_activity):,} companies (any field)")

In [ ]:
# 1d The scope as EDA_SDA_1_Lean cut it (4c); nothing is cut here
print("1d. The scope from EDA_SDA_1_Lean")
display(pd.Series({
    "listings": f"{len(part1):,}",
    "original listings (not reposts)": f"{(~part1.is_repost).sum():,}",
    "companies": f"{part1.company_id.nunique():,}",
    "still open on September 30": f"{part1.live.mean():.1%}",
    "under a TripleTen program": f"{part1.tripleten_program.ne('adjacent tech role').mean():.1%}",
}).to_frame("value"))

**What the data shows**

- **The scope from EDA_SDA_1_Lean: 908K listings (849K originals) from 72K companies;** 24.7% still open on September 30.
- **57.8% are under a TripleTen program;** the rest are adjacent tech roles.

## 2. Analysis views

Some analyses are skewed by part of the table. Each analysis states which **view** it uses:

- **All listings:** everything, reposts included — ranking as an applicant sees it; the repost features.
- **Originals only:** reposts excluded — counts, volumes, windows, company concentration (a repost would double-count).
- **With `posted`:** about 42% of listings — day-of-week patterns, PredictLeads crawler lag.
- **With a salary:** about 48% of listings — pay level and comparisons.
- **Closed listings:** not open on September 30 — simple descriptive durations (survival methods use every listing instead).
- **Long-open listings:** open a long time (3.1's threshold) — ghost vs. desperate-employer checks.
- **Without likely ghosts:** excludes `likely_ghost` (3.1d) — every timing result shown with and without them, from 3.1 on.
- **Open to career changers:** `career_open`, or 3.11's stricter rule — accessibility comparisons, in and outside the scope.
- **The scope:** EDA_SDA_1's in-scope table, by role bucket (3.6) and program — everything in this notebook.
- **Other fields:** EDA_SDA_1's section 7 and occupation summary — comparing the scope with other fields.

### Confounders: compare within groups, don't filter

Some factors travel together, so a raw gap can belong to something else. **Every factor test in section 3 is checked against these,** by comparing within groups (stratifying), not by filtering them out.

- **Company hiring volume** (bands 1–4 / 5–49 / 50–499 / 500+): the strongest signal — every factor compared within bands.
- **Source and platform:** the PredictLeads crawler sees closures differently by platform — checked within source and platform.
- **Staffing agencies** (`is_staffing`): post at volume, repost often — shown with and without them.
- **Role bucket and seniority:** different roles close at different speeds for unrelated reasons — checked within bucket (3.6) and by seniority.
- **Month first seen:** later months had less time to close — survival methods censor still-open listings; month as a split.
- **State** (3.7d): pay-transparency laws vary pay disclosure — checked within state.
- **Company type** (`is_public`, `has_parent`): public companies stay open longer at high volume — shown separately.
- **No description** (`has_desc`, 3.9): closes much faster — shown with and without them.
- **Ghost listings** (`likely_ghost`, 3.1d): inflate days open — timing shown without them too.

**The rule:** each factor shows the raw gap, then the gap within volume band × source type, and within role bucket where there are enough listings. EDA_SDA_4 then confirms the survivors with one model that adjusts for all of these at once.

## 2b. Additional cleaning

**Decided (2026-10-09):** five fixes and the app's filter fields here (salary sanity moved into EDA_SDA_1 section 2, description cleanup to EDA_SDA_3), run before the checks in section 3 so they use clean pay, dates and titles. Same rules as EDA_SDA_1: **fix or flag, never drop**; each step's own cell reports how many listings it changed.

- **Staffing agencies:** flagged by company name or domain words, an "about" text, or (added 2026-10-11) agency-style recruiter titles on 3+ listings — a label and opt-in filter, never hidden by default (an unvalidated text rule, decided 2026-10-13).
- **Salary sanity, standardized to hourly** — moved to EDA_SDA_1 section 2: one hourly pay column (`salary_hourly_usd`) for the whole project, bad units and reversed ranges fixed and flagged (`pay_suspect`) there.
- **Date consistency:** EDA_SDA_1 (section 2) already fixed impossible timelines; here, `posted` more than a year before `first_seen` is flagged (`posted_old`), feeding the ghost-listing check (3.1).
- **Contract types, internships and temporary roles:** contract labels (unified by EDA_SDA_1) plus title words split into employment-type flags and one `job_type` label for the app's filter.
- **Title normalization:** lowercased; brackets, locations, requisition IDs and pay stripped, remote/hybrid tags kept as a flag (`title_clean`); `ai_ml_title` flags AI, machine learning or data-science titles.
- **Multi-location listings:** `n_locations` above 1 → one row per listing for timing; the state list feeds the app's location filter.
- **Filter fields for the app:** role level, work arrangement and schedule, built from `seniority`, contract labels and title words.
- **Description cleanup** — moved to EDA_SDA_3, which loads the text directly; this table keeps only description features (`desc_words`, `desc_hash`, `has_html`) to fit in memory.

**Leakage:** every fix uses only what the listing showed when it was posted, so none of them leaks the outcome.

In [ ]:
# 2b.0 Work on EDA_SDA_1_Lean's table (not a copy: memory); every step logs how many listings it changed (counts only)
jobs = part1
N_START = len(jobs)
del part1
clean_log = []


def log(step, change, n):
    clean_log.append({"step": step, "change": change, "listings": int(n)})

In [ ]:
# 2b.1 Staffing agencies: flag listings posted by recruiters or staffing firms (kept, flagged)
# Company name, domain and "about" text come from the companies table (every company), loaded for this table's
# companies only, one saved part at a time.
ids = set(jobs.company_id.dropna())
CO_COLS = ["id", "attributes.company_name", "attributes.domain", "attributes.description_short", "attributes.meta_title"]
co = (map_parts("companies", lambda d: d.loc[d["id"].isin(ids), d.columns.intersection(CO_COLS)])
      .drop_duplicates("id").set_index("id").reindex(columns=CO_COLS[1:]))

STAFFING_NAME = (r"staffing|recruit|\btalent\b|personnel|\btemps?\b|placement|headhunt|\bsearch (?:group|partners)\b"
                 r"|employment (?:agency|services|solutions)|manpower|robert half|adecco|randstad|insight global"
                 r"|teksystems|aerotek|kforce|apex systems|cybercoders|jobot|allegis|express employment|spherion"
                 r"|akkodis|beacon hill|motion recruitment|kelly services|\bhays\b")
STAFFING_ABOUT = (r"staffing (?:agency|firm|company|services|solutions)|recruit(?:ing|ment) (?:agency|firm|company)"
                  r"|executive search firm|placement (?:agency|firm)|temporary staffing|we place (?:candidates|talent)")


def text_of(*cols):
    """The given company columns joined into one lowercase text per company (blank where missing)."""
    return co.reindex(columns=list(cols)).fillna("").astype(str).agg(" ".join, axis=1).str.lower()


co["by_name"] = text_of("attributes.company_name", "attributes.domain").str.contains(STAFFING_NAME, regex=True)
co["by_about"] = text_of("attributes.description_short", "attributes.meta_title").str.contains(STAFFING_ABOUT, regex=True)
staffing_ids = co.index[co.by_name | co.by_about]
staffing_by_name = jobs.company_id.isin(staffing_ids)

# Recruiter titles (added 2026-10-11): only 10% of busy recruiters' listings came from agencies flagged by name (2b.1b), and
# a staffing firm's recruiters often carry sales titles (account manager, delivery manager, bench sales). Corporate titles
# (recruiter, talent acquisition) are left out: employers use them too. A first-pass keyword rule (decided 2026-10-13: not hand-validated, so a label and opt-in filter, never hidden by default). A company is flagged when at least 3 of its listings,
# and half of those with a recruiter title, name an agency-style title; one listing alone flags nothing.
AGENCY_TITLE = (r"bench sales|staffing|account (?:manager|executive|director|lead)|delivery (?:manager|lead|head|director)"
                r"|client (?:partner|manager|services)|recruitment consultant|resource (?:manager|coordinator)"
                r"|business development (?:manager|executive)")
rtitle = jobs.recruiter_title.astype("string").str.lower().str.strip().replace("", pd.NA)
jobs["agency_recruiter"] = rtitle.str.contains(AGENCY_TITLE, regex=True).fillna(False).astype(bool)
by_co = jobs.loc[rtitle.notna(), ["company_id", "agency_recruiter"]].groupby("company_id").agency_recruiter.agg(["sum", "mean"])
agency_cos = by_co.index[(by_co["sum"] >= 3) & (by_co["mean"] >= 0.5)]
jobs["staffing_by_title"] = jobs.company_id.isin(agency_cos) & ~staffing_by_name
jobs["is_staffing"] = staffing_by_name | jobs.staffing_by_title
log("2b.1", "flagged: posted by a staffing agency or recruiter", jobs.is_staffing.sum())

print("2b.1 Staffing agencies: companies and listings flagged")
display(pd.Series({
    "companies in this table": f"{len(ids):,}",
    "with a name or domain": f"{co['attributes.company_name'].notna().mean():.1%}",
    "with an 'about' text": f"{co['attributes.description_short'].notna().mean():.1%}",
    "flagged by name or domain": f"{co.by_name.sum():,}",
    "flagged by 'about' text only": f"{(co.by_about & ~co.by_name).sum():,}",
    "flagged by recruiter titles only (new)": f"{len(set(agency_cos) - set(staffing_ids)):,}",
    "share of listings flagged by name or 'about' text": f"{staffing_by_name.mean():.1%}",
    "share of listings flagged, recruiter titles included": f"{jobs.is_staffing.mean():.1%}",
}).to_frame("value"))

In [ ]:
# 2b.1-2 Is an agency-style title a staffing sign? Among listings that name a recruiter title: how often each kind comes
# from a company already flagged by name (a check that doesn't use the title rule), and how fast they close
kind = pd.Series(np.select([jobs.agency_recruiter, rtitle.str.contains(r"recruit|talent|sourc", regex=True).fillna(False)],
                           ["agency-style title", "recruiter or talent acquisition"], "other title"), index=jobs.index).where(rtitle.notna())
title_kinds = pd.DataFrame({
    "listings": kind.value_counts(),
    "share at companies flagged by name": staffing_by_name.groupby(kind).mean(),
    "median days open": jobs[["days_open", "live"]].groupby(kind).apply(lambda g: km.median_survival(g.days_open, ~g.live)),
})
print("2b.1 Recruiter titles: agency-style vs. corporate (listings that name a recruiter title)")
display(title_kinds.style.format({"listings": "{:,.0f}", "share at companies flagged by name": "{:.1%}", "median days open": "{:.1f}"}))

by_occupation = jobs.groupby("occupation").agg(listings=("id", "size"), staffing_share=("is_staffing", "mean"))
print("\n2b.1 Staffing share in the 10 largest scope occupations")
display(by_occupation.sort_values("listings", ascending=False).head(10).style.format({"listings": "{:,.0f}", "staffing_share": "{:.1%}"}))

staffing_window = pd.DataFrame({
    "listings": jobs.groupby("is_staffing").size(),
    "median days open": jobs.groupby("is_staffing").apply(lambda g: km.median_survival(g.days_open, ~g.live), include_groups=False),
}).rename(index={True: "staffing agency or recruiter", False: "employer"})
print("\n2b.1 Median days open (Kaplan-Meier): staffing agencies vs. employers")
display(staffing_window.style.format({"listings": "{:,.0f}", "median days open": "{:.1f}"}))

In [ ]:
# 2b.1 review: the companies flagged most often, and the most common recruiter titles, to check the keywords (shows company
# names and titles: this cell is tagged data-rows, so clear its output before committing)
top_flagged = jobs[jobs.is_staffing].company_id.value_counts().head(25)
display(co.loc[top_flagged.index, ["attributes.company_name", "attributes.domain", "by_name", "by_about"]]
        .assign(listings=top_flagged.values))
top_titles = rtitle.value_counts().head(30)
display(pd.DataFrame({"listings": top_titles, "agency-style": top_titles.index.str.contains(AGENCY_TITLE, regex=True),
                      "share at companies flagged by name": staffing_by_name.groupby(rtitle).mean().reindex(top_titles.index)}))

In [ ]:
# 2b.1b Recruiters vs. staffing agencies (EDA_SDA_1_Lean 5f: a named recruiter is linked to faster closing). Is that the
# staffing agencies? Anonymous recruiter keys only; counts and medians.
per_recruiter = jobs.recruiter_key.map(jobs.recruiter_key.value_counts())
recruiter_band = pd.Series(np.select([jobs.recruiter_key.isna(), per_recruiter.ge(5)],
                                     ["no recruiter named", "recruiter with 5+ listings"], "recruiter with 1-4 listings"),
                           index=jobs.index)
rec = jobs[["days_open", "live", "is_staffing"]].assign(recruiter=recruiter_band)

recruiter_window = pd.DataFrame({
    "listings": rec.groupby(["recruiter", "is_staffing"]).size(),
    "median days open": rec.groupby(["recruiter", "is_staffing"]).apply(lambda g: km.median_survival(g.days_open, ~g.live),
                                                                       include_groups=False),
}).rename(index={True: "staffing agency", False: "employer"}, level="is_staffing")
print("2b.1b Median days open (Kaplan-Meier) by recruiter and staffing agency")
display(recruiter_window.style.format({"listings": "{:,.0f}", "median days open": "{:.1f}"}))

flag_share = pd.DataFrame({
    "share flagged by name or 'about' text": staffing_by_name.groupby(rec.recruiter).mean(),
    "share flagged, recruiter titles included": rec.groupby("recruiter").is_staffing.mean()})
print("\n2b.1b Share of each recruiter group posted by staffing agencies, before and after the recruiter-title rule")
display(flag_share.style.format("{:.1%}"))

In [ ]:
# 2b.2 Date consistency: EDA_SDA_1_Lean 2c already removed impossible timelines and blanked posted dates after first_seen.
# What's left: postings dated more than a year before the PL crawler found them (kept, flagged; feeds the ghost check in 3.1)
jobs["posted_old"] = jobs.posted < jobs.first_seen - pd.Timedelta(days=365)
log("2b.2", "flagged posted_old: posted more than a year before first seen", jobs.posted_old.sum())

In [ ]:
# 2b.3 Contract types, split in two (decided 2026-10-11): employment type (flags: several can be true) and, in 2b.6,
# where the work happens. Then job_type, one label for the app's filter (first match wins: internship, temporary,
# contract, regular), from the flags plus title words. EDA_SDA_1_Lean unified the label spellings (161 labels).
title_low = jobs.title.fillna("").str.lower()
contracts = jobs.contract_types.map(as_list)


def any_label(pattern):
    """A listing has a contract label matching the pattern (labels are lowercase, spaces for _ and -)."""
    rx = re.compile(pattern)
    return contracts.map(lambda v: any(rx.search(x) for x in v))


EMPLOYMENT = {
    "is_full_time": r"\bfull time\b",
    "is_part_time": r"\bpart time\b",
    "is_contract": r"\bcontract|\bfreelance\b|\b1099\b",
    "is_temporary": r"\btemp(?:orary)?\b|\bseasonal\b",
    "is_internship": r"\bintern(?:ship)?s?\b",
}
for col, pattern in EMPLOYMENT.items():
    jobs[col] = any_label(pattern)
    log("2b.3", f"{col}: contract label", jobs[col].sum())

JOB_TYPES = {  # first match wins
    "internship": jobs.is_internship | title_low.str.contains(r"\bintern(?:ship)?s?\b|\bco-?op\b", regex=True),
    "temporary": jobs.is_temporary | title_low.str.contains(r"\btemp(?:orary)?\b|\bseasonal\b", regex=True),
    "contract": jobs.is_contract | title_low.str.contains(r"\bcontract(?:or|-to-hire| to hire)?\b|\bc2h\b|\b1099\b", regex=True),
}
jobs["job_type"] = np.select(list(JOB_TYPES.values()), list(JOB_TYPES), default="regular")
for kind in JOB_TYPES:
    log("2b.3", f"job_type = {kind}", (jobs.job_type == kind).sum())

In [ ]:
# 2b.3-2 Employment type and job type: shares and windows
employment_table = pd.DataFrame({
    "share of listings": {c: jobs[c].mean() for c in EMPLOYMENT},
    "full and part time both": {c: (jobs[c] & jobs.is_full_time & jobs.is_part_time).mean() for c in EMPLOYMENT},
})
print("2b.3 Employment type from contract labels (a listing can have several)")
display(employment_table.style.format("{:.1%}"))

job_type_table = pd.DataFrame({
    "share": jobs.job_type.value_counts(normalize=True),
    "median days open": jobs.groupby("job_type").apply(lambda g: km.median_survival(g.days_open, ~g.live), include_groups=False),
})
print("\n2b.3 Job types: share of listings and median days open (Kaplan-Meier)")
display(job_type_table.style.format({"share": "{:.1%}", "median days open": "{:.1f}"}))

combos = contracts.map(lambda v: " + ".join(sorted(v)) if len(v) else "(none)").value_counts(normalize=True)
print("\n2b.3 The most common contract-label combinations")
display(combos.head(12).to_frame("share of listings").style.format("{:.1%}"))

In [ ]:
# 2b.4 Title normalization: title_clean strips tags that aren't the job (the original title stays)
REMOTE_TAG = r"\b(?:remote|hybrid|on-?site|work from home|wfh)\b"
PAY = r"\$\s?[\d,.]+\s?k?(?:\s?[-–]\s?\$?\s?[\d,.]+\s?k?)?(?:\s?(?:/|per|an?)\s?(?:hr|hour|year|yr|annually|annum))?"
REQ_ID = (r"\b(?:req(?:uisition)?|job)(?:\s*(?:id|no\.?|number))?\s*[:#]?\s*[\w-]*\d[\w-]*"
          r"|\bid\s*[:#]?\s*[\w-]*\d[\w-]*|#\s*\d+|\b\d{4,}\b")
CITY_STATE = r"^[a-z .'-]+,\s?[a-z]{2}$"

NOT_THE_JOB = rf"{CITY_STATE[1:-1]}|(?:{REMOTE_TAG[2:-2]})(?:\s+(?:us|usa))?|us|usa|united states|[\W\d]*"


def clean_title(t):
    """A title without the parts that aren't the job: bracketed notes, trailing locations, pay, requisition ids, sign-on bonus."""
    if not isinstance(t, str):
        return t
    t = t.lower()
    t = re.sub(r"[\(\[][^\)\]]*[\)\]]", " ", t)                       # bracketed notes: location, shift, id
    parts = [p.strip() for p in re.split(r"\s+[-–|]\s+", t)]
    parts = parts[:1] + [p for p in parts[1:] if not re.fullmatch(NOT_THE_JOB, p)]  # "- Austin, TX", "| Remote", "- US"
    t = " - ".join(parts)
    for pattern in (PAY, REQ_ID, REMOTE_TAG, r"\bsign[- ]?on(?: bonus)?\b"):
        t = re.sub(pattern, " ", t)
    return re.sub(r"\s+", " ", t).strip(" -–|,:;/")


jobs["title_remote"] = title_low.str.contains(REMOTE_TAG, regex=True)
# AI/ML roles by title (queued 2026-10-10): the hackathon app's audience is AI/ML, and these roles sit across buckets
AI_ML_TITLE = (r"\bmachine learning\b|\bml\b|\bmlops\b|\bai\b|artificial intelligence|\bdeep learning\b|\bllms?\b"
               r"|\bgen(?:erative)? ?ai\b|\bnlp\b|computer vision|\bdata scien")
jobs["title_clean"] = jobs.title.map(clean_title)
log("2b.4", "title_clean differs from the lowercased title", (jobs.title_clean.fillna("") != jobs.title_norm.fillna("")).sum())
jobs["ai_ml_title"] = jobs.title_clean.fillna("").str.contains(AI_ML_TITLE, regex=True)
log("2b.4", "ai_ml_title: the title names AI, machine learning or data science", jobs.ai_ml_title.sum())

# Would matching reposts on the cleaned title find more of them than EDA_SDA_1_Lean's lowercased title did?
key = ["company_id", "title_clean", "desc_hash"]
with_key = jobs.dropna(subset=key).sort_values("first_seen")
reposts_clean = with_key.duplicated(key).sum()
print("2b.4 Title normalization")
display(pd.Series({
    "distinct titles (lowercased, EDA_SDA_1_Lean)": f"{jobs.title_norm.nunique():,}",
    "distinct titles (title_clean)": f"{jobs.title_clean.nunique():,}",
    "titles changed": f"{(jobs.title_clean.fillna('') != jobs.title_norm.fillna('')).mean():.1%}",
    "titles with a remote or hybrid tag": f"{jobs.title_remote.mean():.1%}",
    "AI, machine learning or data science in the title": f"{jobs.ai_ml_title.mean():.1%}",
    "reposts (EDA_SDA_1_Lean)": f"{jobs.is_repost.sum():,}",
    "reposts if matched on title_clean": f"{reposts_clean:,}",
}).to_frame("value"))

print("\n2b.4 AI/ML titles by role bucket (share of the bucket's listings)")
display(jobs.groupby("role_bucket").ai_ml_title.mean().sort_values(ascending=False).to_frame("AI/ML title").style.format("{:.1%}"))

In [ ]:
# 2b.5 Multi-location listings: one row each for timing; the state list for the app's filter comes with 3.7
jobs["multi_location"] = jobs.n_locations.fillna(1) > 1
log("2b.5", "flagged multi_location: names more than one location", jobs.multi_location.sum())

In [ ]:
# 2b.6 Filter fields for the app: role level, work mode, schedule (job type is 2b.3; location comes with 3.7)
# PredictLeads' seniority follows LinkedIn's levels, where "mid_senior" is one bucket for every experienced individual
# contributor. Decided 2026-10-11: it stays whole (it's the source's label, and title words are a heuristic); a separate
# flag, senior_title (the Profiles rule: senior, sr., lead, principal, staff, architect), carries the title signal, and
# 3.7c and the screening (3.13) test whether it matters.
LEADERSHIP = ["director", "head", "vice_president", "president", "c_level", "founder", "partner"]
title_low = jobs.title.fillna("").str.lower()
seniority = jobs.seniority.fillna("not_set")
jobs["senior_title"] = title_low.str.contains(RULES["senior_target"], regex=True)
jobs["level"] = np.select(
    [seniority.eq("junior"), seniority.eq("mid_senior"), seniority.eq("manager"), seniority.isin(LEADERSHIP)],
    ["entry", "mid_senior", "manager", "leadership"], default="not stated")
log("2b.6", "senior_title: the title says senior, lead, principal, staff or architect", jobs.senior_title.sum())

# Where the work happens (work_arrangement, decided 2026-10-11): contract labels ("fully remote", "100% remote",
# "hybrid", "on site"...) and title words; several signs make "several". The description's remote wording is a check.
title_says = lambda pattern: title_low.str.contains(pattern, regex=True)
places = pd.DataFrame({
    "remote": any_label(r"remote") | title_says(r"\bremote\b|work from home|\bwfh\b"),
    "hybrid": any_label(r"hybrid") | title_says(r"\bhybrid\b"),
    "onsite": any_label(r"\bon ?site\b|\bin office\b") | title_says(r"\bon-?site\b"),
})
n_places = places.sum(axis=1)
jobs["work_arrangement"] = np.select([n_places > 1, places.remote, places.hybrid, places.onsite],
                                     ["several", "remote", "hybrid", "onsite"], default="not stated")
jobs["schedule"] = np.select(
    [jobs.is_full_time & jobs.is_part_time, jobs.is_full_time, jobs.is_part_time],
    ["full or part time", "full time", "part time"], default="not stated")
log("2b.6", "work_arrangement stated (remote, hybrid, onsite or several)", (jobs.work_arrangement != "not stated").sum())
log("2b.6", "schedule stated (full or part time)", (jobs.schedule != "not stated").sum())

In [ ]:
# 2b.6-2 The app's filter fields: share of listings and median days open for each value
def field_table(col, order):
    """For one filter field: each value's share of listings and Kaplan-Meier median days open, in the given order."""
    return pd.DataFrame({
        "share": jobs[col].value_counts(normalize=True),
        "median days open": jobs.groupby(col).apply(lambda g: km.median_survival(g.days_open, ~g.live), include_groups=False),
    }).reindex([o for o in order if o in set(jobs[col])])


FIELD_FORMATS = {"share": "{:.1%}", "median days open": "{:.1f}", "share with a senior title": "{:.1%}",
                 "share whose description mentions remote": "{:.1%}"}
level_table = field_table("level", ["entry", "mid_senior", "manager", "leadership", "not stated"]).assign(
    **{"share with a senior title": jobs.groupby("level").senior_title.mean()})
print("2b.6 Role level (entry = junior; mid_senior as PredictLeads labels it; leadership = director and above)")
display(level_table.style.format(FIELD_FORMATS))

arrangement_table = field_table("work_arrangement", ["remote", "hybrid", "onsite", "several", "not stated"]).assign(
    **{"share whose description mentions remote": jobs.groupby("work_arrangement").mentions_remote.mean()})
print("\n2b.6 Work arrangement (with the share whose description mentions remote, as a check)")
display(arrangement_table.style.format(FIELD_FORMATS))

print("\n2b.6 Schedule")
display(field_table("schedule", ["full time", "part time", "full or part time", "not stated"]).style.format(FIELD_FORMATS))

In [ ]:
# 2b.6-3 Chart: the app's filter fields in the scope (share of listings)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (col, title) in zip(axes, [("level", "Role level"), ("work_arrangement", "Work arrangement"), ("schedule", "Schedule"),
                                   ("job_type", "Job type")]):
    bar_h(ax, 100 * jobs[col].value_counts(normalize=True), title, "% of listings", "{:.0f}%")
plt.tight_layout()
plt.show()

In [ ]:
# 2b.7 Summary: what each cleaning step changed (no listing removed)
print("2b. Additional cleaning: listings changed or flagged by each step")
display(pd.DataFrame(clean_log).set_index(["step", "change"]).style.format({"listings": "{:,.0f}"}))
print(f"listings: {N_START:,} before, {len(jobs):,} after (fix or flag, never drop)")

**What the data shows**

- **Staffing agencies are a small, fast group:** 2.0% of listings (753 companies), closing in a median 12.6 days vs. 24.0 for employers. **The new recruiter-title rule adds 33 companies:** agency-style titles (account, delivery or client manager) close in 6.2 days and come from name-flagged agencies about twice as often as corporate recruiter titles (13% vs. 6.8%). It doubles the share of busy recruiters' listings flagged (10% → 21%) but barely moves the total (1.9% → 2.0%); a first-pass keyword rule, not hand-checked (decided 2026-10-13): a label ("staffing agency, from the title") and an opt-in filter, not something that hides or demotes on its own. **Recruiters aren't just staffing agencies:** among employers, recruiters with 5+ listings still close in 8.1 days vs. 24.1 with none named.
- **Contract labels:** 55% full time, 9.7% contract, 3.8% part time, 3.3% internship, 17.5% none. Full and part time together is rare (1.2%). Where the work happens is stated for 54% (remote 17%, onsite 15%, hybrid 13%, several 8.3%), and remote labels agree with the description 94% of the time.
- **Titles and levels:** cleaning changed 25% of titles; **9.0% name AI, machine learning or data science** (25% of data science, 14% of networks and systems). Role levels: mid_senior 77% (kept whole; 40% of it has a senior title, flagged `senior_title`), entry 10%, manager 9.4%, leadership 3.7%. No listing names more than one location: PredictLeads lists each location separately.

## 3. Checks and analyses

Record each result and the decision it leads to here, then apply it when building the final table (section 4). **From 3.1 on, every timing result is also shown without likely ghosts** (section 2's views). **Every factor test follows the confounders rule in section 2:** raw gap, then the gap within volume band × source type (and role bucket where possible); only gaps that hold count.

**3.13 closes the section with the feature screening:** every candidate feature through the same checks, one keep/drop table.

In [ ]:
# 3.0 Display helpers used through section 3 (written out here, replacing src/eda_helpers.py's heatmap and show_shaded)
def heatmap(ax, table, title, fmt="{:,.0f}", cmap="Blues", vmin=None, vmax=None):
    """An annotated heatmap of a small table: each cell's value written in it, darker = higher (vmin and vmax fix the
    colour scale; a scale that runs from below zero to above it, like a correlation, is read as diverging)."""
    values = table.values.astype(float)
    ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(table.shape[1]), [str(c)[:28] for c in table.columns], rotation=35, ha="right", fontsize=8.5)
    ax.set_yticks(range(table.shape[0]), [str(i)[:34] for i in table.index], fontsize=8.5)
    finite = values[np.isfinite(values)]
    low = vmin if vmin is not None else (finite.min() if finite.size else 0)
    high = vmax if vmax is not None else (finite.max() if finite.size else 1)
    diverging = low < 0 < high
    for (row, col), value in np.ndenumerate(values):
        if np.isfinite(value):
            depth = abs(value) / max(abs(low), abs(high)) if diverging else (value - low) / ((high - low) or 1)  # how dark the cell is
            ax.text(col, row, fmt.format(value), ha="center", va="center", fontsize=8, color="white" if depth > 0.6 else INK)
    ax.set_title(title)
    ax.grid(False)


def format_rows(table, formats):
    """A copy of the table as text where each ROW has its own format (formats: row name -> format string), for tables
    whose rows are different measures (a pay, a count, a share) so no single column format fits."""
    shown = table.astype(object)
    for row, fmt in formats.items():
        for col in table.columns:
            value = table.loc[row, col]
            shown.loc[row, col] = fmt.format(value) if pd.notna(value) else ""
    return shown


def shade(table, formats, highlight=None):
    """Display a table: one format per column (a dict, or one string for every column), each numeric column shaded on
    its own scale, and the row named `highlight` outlined."""
    styled = table.style.format(formats)
    if highlight in table.index:
        outline = f"font-weight: bold; border-top: 2px solid {ORANGE}; border-bottom: 2px solid {ORANGE}"
        styled = styled.apply(lambda row: [outline if row.name == highlight else "" for _ in row], axis=1)
    for c in table.columns:
        if pd.api.types.is_numeric_dtype(table[c]) and table[c].notna().sum() > 1 and table[c].nunique() > 1:
            styled = styled.background_gradient(cmap="Blues", subset=[c], low=0.05, high=0.35)
    display(styled)

In [ ]:
# 3.0 Section 3 setup: each listing's role bucket (O*NET code; detailed 15-1299 codes first) and the counting view
jobs["role_bucket"] = jobs.onet_code.map(role_bucket).fillna("unmapped")
# O*NET 13-1161 split by title (decided 2026-10-11): analyst, research and insights titles stay in business and data
# analysis; marketing specialists get their own label (src/career_switch.py). A first-pass keyword rule (decided 2026-10-13: not hand-validated, so a label and opt-in filter, never hidden by default).
is_marketing = marketing_specialist(jobs.onet_code, jobs.title_clean).to_numpy()
jobs.loc[is_marketing, "role_bucket"] = MARKETING_BUCKET
# Occupations hidden in the app (kept in the data): document management specialists are records work, not a tech path.
# Marketing specialists are labelled (role bucket), not hidden: the title split is an unvalidated keyword rule.
HIDDEN_OCCUPATIONS = {"15-1299.03": "document management specialists"}
jobs["hidden_occupation"] = jobs.onet_code.fillna("").str[:10].isin(HIDDEN_OCCUPATIONS)
orig = jobs[~jobs.is_repost]  # counts and sizes use each job once (reposts excluded); flags stay on every listing

# Company hiring volume from each company's EARLIER listings in any field (EDA_SDA_1_Lean 5c), banded; bands count the
# listing itself. Rescaled for the full file (EDA_SDA_1_Lean 5c): 1-4 / 5-49 / 50-499 / 500+.
VOL_BANDS = ["1-4", "5-49", "50-499", "500+"]
jobs["volume_band"] = pd.cut(jobs.company_earlier + 1, [0, 4, 49, 499, np.inf], labels=VOL_BANDS)
# Weekly rates use full weeks only (EDA_SDA_1_Lean 6a: the first and last weeks are partial)
FULL_WEEKS = jobs.loc[jobs.full_week, "first_seen"].dt.tz_localize(None).dt.to_period("W").nunique()


def km_by(df, by):
    """Kaplan-Meier median days open for each group (still-open listings censored). Groups only the two columns it
    needs, so a million-row table isn't copied once per group."""
    keys = [df[b] for b in ([by] if isinstance(by, str) else by)]
    return df[["days_open", "live"]].groupby(keys, observed=True).apply(lambda g: km.median_survival(g.days_open, ~g.live))

### 3.1 Ghost listings vs. desperate employers

A long-open listing may be a **ghost listing** (forgotten or no longer monitored: rank lower or flag it) or a **desperate employer** (still actively hiring: rank higher). Age alone can't tell them apart.

| Signal | Points to "still hiring" | Points to "ghost listing" |
|:--|:--|:--|
| Relisted (`times_listed`, `days_since_repost_last_listed`) | Relisted recently | Never relisted in months |
| Company still posting new listings | The company is actively recruiting | The company stopped posting long ago |
| Source | Hiring platforms often auto-expire listings, so a long-open one was likely kept on purpose | Company career pages can host forgotten listings indefinitely |
| `posted` vs. `first_seen` | Recently refreshed | Very old posting date |
| `last_processed_at` (meaning to confirm with PredictLeads) | May record content changes | |

**Decided (2026-10-10): hide, never remove.** **Likely ghosts** (older than **90 days** on the date being viewed and from a company gone quiet, 3.1d) are hidden in the app (a note says how many, and why); other listings older than 90 days stay visible, labelled "open 90+ days" and ranked lower; every listing stays in the analysis and the model, with still-open listings censored as usual. Age on a date is known at that moment, so hiding by it uses no future information. Removing listings by how long they *ended up* open would bias the timing model (it would never see a long window) and throw away struggling employers, who may be the most open to a career changer.

Steps: describe the long-open listings (how many, which occupation groups and sources); **check the 90-day threshold** (do closing rates level off past 90 days? are those listings relisted, or did their company stop posting?) and keep it or adjust it with that evidence; measure each signal; combine them into a **"likely ghost listing" flag**, used to rank lower or label, never to remove. Only listings first seen before July can reach 90 days in this six-month window. Evergreen wording ("always accepting applications") is added in EDA_SDA_3.

In [ ]:
# 3.1a How long do listings stay open, and does closing slow down past 90 days? (Kaplan-Meier, original listings)
# Only listings first seen in April-May can be followed for 120+ days in this window, so the curve uses them.
DATA_END = pd.Timestamp("2026-09-30", tz="UTC")
STALE_DAYS = 90
early = orig[orig.first_seen < pd.Timestamp("2026-06-01", tz="UTC")]
curve = km.km_curve(early.days_open, ~early.live)
closing_by_age = {}
for t in [0, 30, 60, 90, 120]:
    open_at_start, open_at_end = km.survival_at(curve, t), km.survival_at(curve, t + 30)
    closing_by_age[f"day {t} to {t + 30}"] = {
        "still open at the start": open_at_start,
        "of those, share that close in the next 30 days": 1 - open_at_end / open_at_start if open_at_start > 0 else np.nan}
print(f"3.1a Closing by listing age ({len(early):,} listings first seen in April-May)")
display(pd.DataFrame(closing_by_age).T.style.format("{:.1%}"))

In [ ]:
# 3.1b Long-open listings on September 30: still open and more than 90 days old (labelled "open 90+ days" in the app; likely ghosts among them are hidden)
age = (DATA_END - orig.first_seen).dt.days
long_open = orig.live & (age > STALE_DAYS)
open_by_bucket = pd.DataFrame({
    "still open": orig.groupby("role_bucket").live.sum(),
    f"older than {STALE_DAYS} days": long_open.groupby(orig.role_bucket).sum(),
    "share of open": long_open.groupby(orig.role_bucket).sum() / orig.groupby("role_bucket").live.sum(),
}).sort_values("still open", ascending=False)
print(f"3.1b Listings still open on September 30 and older than {STALE_DAYS} days")
display(open_by_bucket.style.format({"still open": "{:,.0f}", f"older than {STALE_DAYS} days": "{:,.0f}", "share of open": "{:.1%}"}))

open_by_source = pd.DataFrame({
    "still open": orig.groupby("source_type").live.sum(),
    "share older than 90 days": long_open.groupby(orig.source_type).sum() / orig.groupby("source_type").live.sum()})
print("\n3.1b The same, by source type")
display(open_by_source.style.format({"still open": "{:,.0f}", "share older than 90 days": "{:.1%}"}))

In [ ]:
# 3.1c Ghost or still hiring? Signals for open listings, long-open vs. younger (shares; all as of September 30)
# - company still posting: the company's newest listing in any field (saved in 1c) is from the last 60 days
# - relisted: the same job (company, clean title, description) appears more than once
# - very old posting date: posted more than a year before first seen (2b.2)
QUIET_DAYS = 60
job_key = ["company_id", "title_clean", "desc_hash"]
copies = jobs.dropna(subset=job_key).groupby(job_key).id.transform("size").reindex(jobs.index).fillna(1)
signals = pd.DataFrame({
    "company still posting": (DATA_END - jobs.company_id.map(company_last_new)).dt.days <= QUIET_DAYS,
    "relisted": copies > 1,
    "on a company site or other": jobs.source_type.eq("company site / other"),
    "very old posting date": jobs.posted_old.fillna(False),
    "staffing agency": jobs.is_staffing,
})
older = orig.index[long_open]
younger = orig.index[orig.live & ~long_open]
print("3.1c Signals among open listings: older than 90 days vs. 90 days or younger")
display(pd.DataFrame({
    f"older than {STALE_DAYS} days ({len(older):,})": signals.loc[older].mean(),
    f"{STALE_DAYS} days or younger ({len(younger):,})": signals.loc[younger].mean(),
}).style.format("{:.1%}"))

In [ ]:
# 3.1d A first ghost-listing flag (provisional; evergreen wording joins it in EDA_SDA_3). Used to label or rank lower,
# never to remove. Among listings open and older than 90 days:
# - likely ghost: the company has posted nothing new in 60 days, or the posting date is more than a year old
# - possibly hard to fill: the company is still posting. Tested below and dropped as a category on 2026-10-10: these
#   listings are no more open to newcomers than younger ones, so stale listings that aren't likely ghosts are just "stale"
jobs["age_on_sep30"] = (DATA_END - jobs.first_seen).dt.days
jobs["stale"] = jobs.live & (jobs.age_on_sep30 > STALE_DAYS)
quiet = ~signals["company still posting"]
jobs["likely_ghost"] = jobs.stale & (quiet | signals["very old posting date"])
jobs["possibly_hard_to_fill"] = jobs.stale & ~jobs.likely_ghost
o = jobs.loc[orig.index]
ghosts_by_bucket = pd.DataFrame({
    "stale (open, older than 90 days)": o.groupby("role_bucket").stale.sum(),
    "likely ghost": o.groupby("role_bucket").likely_ghost.sum(),
    "possibly hard to fill": o.groupby("role_bucket").possibly_hard_to_fill.sum(),
}).sort_values("stale (open, older than 90 days)", ascending=False)
print("3.1d Long-open listings: likely ghosts vs. possibly hard to fill (original listings)")
display(ghosts_by_bucket.style.format("{:,.0f}"))
print(f"stale: {o.stale.sum():,} of {o.live.sum():,} open listings | likely ghosts: {o.likely_ghost.sum():,} | "
      f"possibly hard to fill: {o.possibly_hard_to_fill.sum():,}")

# Is "possibly hard to fill" worth keeping? Only if those listings are more open to newcomers than the rest
o = o.assign(group=np.select([o.likely_ghost, o.possibly_hard_to_fill, o.live],
                             ["likely ghost", "possibly hard to fill", "open, 90 days or younger"], default="closed"),
             is_entry=o["level"].eq("entry"), is_open_to_changers=o["career_open"].astype(float))
openness = o[o["group"] != "closed"].groupby("group").agg(listings=("group", "size"), entry_level=("is_entry", "mean"),
                                                         open_to_career_changers=("is_open_to_changers", "mean"))
print("\n3.1d Openness to career changers: possibly hard to fill vs. likely ghosts vs. younger open listings")
display(openness.style.format({"listings": "{:,.0f}", "entry_level": "{:.1%}", "open_to_career_changers": "{:.1%}"}))

#### 3.1e The career-changer view

The app ranks only what a career changer can see, so the key results are also shown on that view: **listings visible to a newcomer without management experience** (no manager level or above, no likely ghost, no hidden occupation). Senior-titled listings stay in it, flagged (`senior_title`): **decided 2026-10-11, the app shows them ranked lower with a "senior" label rather than hiding them** (81% of them ask for 5+ years and 4.9% show openness, vs. 43% and 17% for other mid_senior listings, 3.7c; but a title is a rough signal, and some career changers have senior experience). A third column shows the view without them. The feature screening (3.13) uses this view first.

**Result:** the view keeps 86% of the scope's original listings and looks like the whole scope (25% still open, a median 24.1 days, 29% closed within a week), paying a little less ($61.92 vs. $64.62 an hour). Without senior titles it keeps 55%: lower pay ($50.67), more entry level (17%) and more openness to career changers (34% vs. 22%), with similar timing.

In [ ]:
# 3.1e The career-changer view: what a newcomer without management experience sees (a flag; nothing is removed)
jobs["cc_visible"] = ~jobs.likely_ghost & ~jobs.hidden_occupation & ~jobs.level.isin(["manager", "leadership"])
o = jobs.loc[orig.index]


def view_summary(g):
    """One column of the comparison: the same measures for one view of the scope."""
    curve = km.km_curve(g.days_open, ~g.live)
    return pd.Series({
        "share of the scope's original listings": len(g) / len(o), "still open on September 30": g.live.mean(),
        "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live), "closed within 7 days": 1 - km.survival_at(curve, 7),
        "median hourly pay": g.salary_hourly_usd.median(), "entry level": g.level.eq("entry").mean(),
        "senior title": g.senior_title.mean(), "open to career changers (any sign)": g.career_open.astype(float).mean(),
        "remote": g.work_arrangement.eq("remote").mean(), "from companies with 500+ listings": g.volume_band.eq("500+").mean()})


views = pd.DataFrame({
    "whole scope": view_summary(o), "career-changer view": view_summary(o[o.cc_visible]),
    "career-changer view, no senior titles": view_summary(o[o.cc_visible & ~o.senior_title])})
VIEW_FORMATS = {"share of the scope's original listings": "{:.1%}", "still open on September 30": "{:.1%}",
                "median days open (Kaplan-Meier)": "{:.1f}", "closed within 7 days": "{:.1%}", "median hourly pay": "${:,.2f}",
                "entry level": "{:.1%}", "senior title": "{:.1%}", "open to career changers (any sign)": "{:.1%}",
                "remote": "{:.1%}", "from companies with 500+ listings": "{:.1%}"}
print("3.1e The whole scope vs. the career-changer view (original listings)")
display(format_rows(views, VIEW_FORMATS))

In [ ]:
# 3.1-chart How closing slows with age (left), and stale listings by role bucket (right)
ages = list(range(0, 150, 30))
close_next = [100 * (1 - km.survival_at(curve, t + 30) / km.survival_at(curve, t)) for t in ages]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
ax1.bar([f"{t}-{t + 30}" for t in ages], close_next, color=[BLUE if t < 90 else ORANGE for t in ages], width=0.6)
for x, v in enumerate(close_next):
    ax1.text(x, v, f"{v:.0f}%", ha="center", va="bottom", color=INK_2, fontsize=8.5)
ax1.set(title="Share of open listings that close in the next 30 days", xlabel="listing age (days)", ylabel="%")
ax1.grid(axis="x", visible=False)

oo = jobs.loc[orig.index]
stale_by = pd.DataFrame({"likely ghost (hidden)": oo.groupby("role_bucket").likely_ghost.sum(),
                         "other stale (labelled)": oo.groupby("role_bucket").apply(
                             lambda g: (g.stale & ~g.likely_ghost).sum(), include_groups=False)})
stale_by = stale_by[stale_by.sum(axis=1) > 0].sort_values("likely ghost (hidden)")
left = np.zeros(len(stale_by))
for col, color in [("likely ghost (hidden)", ORANGE), ("other stale (labelled)", MUTED)]:
    ax2.barh(stale_by.index, stale_by[col], left=left, color=color, height=0.65, label=col)
    left += stale_by[col].values
ax2.set(title="Open listings older than 90 days, by role bucket", xlabel="listings")
ax2.legend(fontsize=8, frameon=False)
ax2.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**What the data shows**

- **Closing slows with age:** of listings first seen in April-May, 53% close in their first 30 days; past 60 days, 30-36% of those still open close each month. Old listings still close, so age alone isn't a ghost.
- **19% of open listings are older than 90 days** (40K of 216K); open job-board listings are almost all old (95%), since job boards stop after May (3.2c).
- **7.5K likely ghosts** (0.9% of the scope, 3.5% of open listings): stale and from a company with no new listing in 60 days, or with a posting date over a year old. The other stale listings are no more open to newcomers (16.5% vs. 20.5% for younger open listings), so "possibly hard to fill" is dropped as a label.

### 3.2 Source / platform

| Check | If the result is… | Then… |
|:--|:--|:--|
| Median days open by source, *within* each company-volume band | Source gaps hold within every band | Keep source as its own factor |
| | Gaps mostly disappear | Drop it, or keep it as a minor factor |
| Share of job-board listings whose company + title also appears on a company site or platform | High (copies) | Drop job-board rows to avoid double-counting |
| | Low | Keep them |
| Platform sizes in the chosen groups | Only a few platforms have 1,000+ listings | Name those platforms; group the rest as "other platform" |

**Also (added 2026-10-10):** why every open job-board listing is over 90 days old (3.1b); the domain check compares each listing's web domain with its company's domain (loaded for the scope only, never printed).

In [ ]:
# 3.2a How big is each source? Source type and named platforms (original listings)
o = jobs.loc[orig.index]
source_sizes = pd.DataFrame({
    "listings": o.source_type.value_counts(),
    "share": o.source_type.value_counts(normalize=True),
    "still open": o.groupby("source_type").live.sum(),
})
print("3.2a Source types in the scope (original listings)")
display(source_sizes.style.format({"listings": "{:,.0f}", "share": "{:.1%}", "still open": "{:,.0f}"}))

platforms = o.platform.value_counts()
print("\n3.2a Named hiring platforms and job boards with 100+ listings (the rest are grouped as 'other platform')")
display(platforms[platforms >= 100].rename_axis("platform").to_frame("listings").style.format("{:,.0f}"))
print(f"other platforms (under 100 listings each): {platforms[platforms < 100].size} platforms, {platforms[platforms < 100].sum():,} listings")

In [ ]:
# 3.2b Does source matter beyond company volume? Median days open (Kaplan-Meier) by source type within each volume
# band (the confounders rule), with and without likely ghosts. A gap that holds in every band is source, not volume.
gap_tables = {}
for name, df in {"all listings": o, "without likely ghosts": o[~o.likely_ghost], "career-changer view": o[o.cc_visible]}.items():
    t = km_by(df, ["volume_band", "source_type"]).unstack("source_type").reindex(VOL_BANDS)
    t["listings"] = df.groupby("volume_band", observed=True).size()
    gap_tables[name] = t
    print(f"3.2b Median days open (Kaplan-Meier) by source type within company-volume bands ({name})")
    display(t.style.format({**{c: "{:.1f}" for c in t.columns if c != "listings"}, "listings": "{:,.0f}"}))

named = platforms[platforms >= 100].index
if len(named):
    print("\n3.2b Median days open (Kaplan-Meier) by named platform (100+ listings)")
    display(km_by(o[o.platform.isin(named)], "platform").sort_values().rename_axis("platform").to_frame("median days open")
            .style.format("{:.1f}"))

In [ ]:
# 3.2c Why is every open job-board listing over 90 days old (3.1b)? When job-board listings were first seen, and how
# many are still open, by month
by_month = pd.crosstab(o.loc[~o.partial_month, "month"], o.loc[~o.partial_month, "source_type"])
print("3.2c Listings first seen per month, by source type (full months)")
display(by_month.style.format("{:,.0f}"))

job_boards = o[o.source_type == "job board"]
print("\n3.2c Job-board listings: first seen by month, and how many are still open")
display(pd.DataFrame({"listings": job_boards.groupby("month").size(), "still open": job_boards.groupby("month").live.sum()})
        .style.format("{:,.0f}"))

In [ ]:
# 3.2d Domain check: is a "company site / other" listing really on the employer's own website?
# Each listing's web domain (from its URL, in the in-scope table; never printed) is compared with its company's domain
# (2b.1). Only counts are shown.
from urllib.parse import urlsplit


def host(url):
    """The web host of a URL, lowercase and without "www." ("" if it can't be read)."""
    try:
        h = urlsplit(url).hostname or ""
    except ValueError:
        return ""
    return h.lower().removeprefix("www.")


company_domain = jobs.company_id.map(co["attributes.domain"].fillna("").str.lower().str.removeprefix("www.")).fillna("")
listing_host = jobs.url.map(host).fillna("")
jobs["on_employer_domain"] = [bool(d) and (h == d or h.endswith("." + d))  # careers.acme.com counts for acme.com
                              for h, d in zip(listing_host, company_domain)]
o = jobs.loc[orig.index]
site = o[o.source_type == "company site / other"]
del listing_host, company_domain
print("3.2d 'Company site / other' listings: on the employer's own domain or not (original listings)")
display(pd.Series({
    "company site / other listings": f"{len(site):,}",
    "share on the employer's own domain": f"{site.on_employer_domain.mean():.1%}",
    "share on another site (unlisted platforms, aggregators, career-site hosts)": f"{(~site.on_employer_domain).mean():.1%}",
    "share of all in-scope listings on the employer's own domain": f"{o.on_employer_domain.mean():.1%}",
}).to_frame("value"))

In [ ]:
# 3.2-chart Median days open by source x company volume (left) and by named platform (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.2]})
heatmap(ax1, gap_tables["all listings"].drop(columns="listings").T, "Median days open (Kaplan-Meier): source x company volume",
        fmt="{:.0f}", cmap="Blues")
if len(named):
    bar_h(ax2, km_by(o[o.platform.isin(named)], "platform"), "Median days open by platform (100+ listings)", "days", "{:.0f}")
plt.tight_layout()
plt.show()

**What the data shows**

- **Source matters beyond company volume, and the two interact:** at companies with 500+ listings, company-site listings close in a median 8 days vs. 22 on hiring platforms; at companies with 1-4, 64 vs. 47. Job boards sit at 12-18 days in every band.
- **Platforms differ widely:** Built In, Indeed, SmartRecruiters, Taleo, Oracle and LinkedIn close in 10-16 days; Greenhouse, Breezy, Lever and Ashby in 49-72, partly in how they show closures to the PredictLeads crawler.
- **Job boards stop after May** (about 30K listings a month in April-May, about 200 a month after): a PredictLeads collection change to ask them about. **36.0% of all in-scope listings are on the employer's own domain,** so the app can link many straight to the employer.

### 3.3 Freshness and PredictLeads crawler lag

| Check | If the result is… | Then… |
|:--|:--|:--|
| Share of listings found more than 7, 14 and 30 days after `posted` | Tiny | Use `first_seen` only |
| | Meaningful | Add a "found late" flag |
| Survival curves: found late vs. found promptly | Found-late listings close much sooner | The flag matters for apply-by estimates |
| Survival curves: listings with vs. without a `posted` date | Similar | `posted` can be used where present |
| | Different | Stick with `first_seen` only |

**Also (added 2026-10-10): are high-volume companies crawled more often?** The PredictLeads crawler's visit times aren't in the data, so PredictLeads crawler lag (posting date → first seen) stands in for crawl frequency, and the company-volume effect from 3.5a (12 vs. 44 days) is re-measured from the employer's posting date. If it shrinks a lot, part of it was crawl timing.

In [ ]:
# 3.3a Crawler lag: days from the employer's posting date to the PL crawler first finding the listing
# (listings with `posted`, about 44%; posting dates after first seen were blanked in EDA_SDA_1_Lean 2c)
o = jobs.loc[orig.index]
with_posted = o[o.posted.notna()].copy()
with_posted["lag_days"] = (with_posted.first_seen - with_posted.posted).dt.total_seconds() / 86400
lag = with_posted.lag_days.clip(lower=0)


def lag_summary(s):
    """One row of the lag table: how long the PL crawler took to find a group's listings."""
    return pd.Series({"listings": len(s), "median lag (days)": s.median(), "found within 1 day": (s <= 1).mean(),
                      "within 7 days": (s <= 7).mean(), "30+ days late": (s > 30).mean()})


lag_table = pd.DataFrame({
    "all": lag_summary(lag),
    **{st: lag_summary(lag[with_posted.source_type == st]) for st in ["company site / other", "hiring platform", "job board"]},
}).T
print("3.3a Crawler lag (listings with a posting date)")
display(lag_table.style.format({"listings": "{:,.0f}", "median lag (days)": "{:.1f}", "found within 1 day": "{:.1%}",
                                "within 7 days": "{:.1%}", "30+ days late": "{:.1%}"}))

In [ ]:
# 3.3b Are high-volume companies crawled more often? A shorter lag means the PL crawler visits more often.
lag_by_volume = (with_posted.assign(lag=lag).groupby(["source_type", "volume_band"], observed=True).lag.median()
                 .unstack("volume_band").reindex(columns=VOL_BANDS))
print("3.3b Crawler lag by company volume, within source type (median days; listings with a posting date)")
display(lag_by_volume.style.format("{:.1f}"))

In [ ]:
# 3.3c Does the volume effect survive measuring from the employer's posting date? For the same listings, the window
# from first seen (days_open) vs. the window from the posting date (last seen - posted). If the 1-4 vs. 500+ gap
# shrinks a lot, part of it was crawl timing, not hiring speed.
with_posted["days_from_posted"] = (with_posted.last_seen - with_posted.posted).dt.total_seconds() / 86400
from_posted = with_posted.assign(days_open=with_posted.days_from_posted)
window_by_volume = pd.DataFrame({
    "listings": with_posted.groupby("volume_band", observed=True).size(),
    "from first seen": km_by(with_posted, "volume_band"),
    "from the posting date": km_by(from_posted, "volume_band"),
    "from the posting date, without likely ghosts": km_by(from_posted[~from_posted.likely_ghost], "volume_band"),
}).reindex(VOL_BANDS)
print("3.3c Median window (Kaplan-Meier) by company volume, from first seen vs. from the posting date (same listings)")
display(window_by_volume.style.format({"listings": "{:,.0f}", "from first seen": "{:.1f}", "from the posting date": "{:.1f}",
                                       "from the posting date, without likely ghosts": "{:.1f}"}))

In [ ]:
# 3.3d Late-found listings: found more than 7 days after posting vs. promptly (within 1 day), median windows
found = np.select([lag <= 1, lag > 7], ["found within 1 day", "found 7+ days late"], default="found in 1-7 days")
late_vs_prompt = pd.DataFrame({
    "listings": pd.Series(found).value_counts(),
    "from first seen": km_by(with_posted.assign(g=found), "g"),
    "from the posting date": km_by(from_posted.assign(g=found), "g"),
})
print("3.3d Late-found vs. prompt listings (Kaplan-Meier medians; listings with a posting date)")
display(late_vs_prompt.style.format({"listings": "{:,.0f}", "from first seen": "{:.1f}", "from the posting date": "{:.1f}"}))

In [ ]:
# 3.3e Chart: survival curves for companies with 1-4 listings vs. 500+, measured from first seen and from
# the employer's posting date (listings with a posting date). Where each curve crosses the dashed 50% line is its
# Kaplan-Meier median; the gap between the curves is the company-volume effect.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (title, col) in zip(axes, [("From first seen", "days_open"), ("From the employer's posting date", "days_from_posted")]):
    for band, color in [("1-4", MUTED), ("500+", BLUE)]:
        g = with_posted[with_posted.volume_band == band]
        band_curve = km.km_curve(g[col], ~g.live)
        ax.step(band_curve.index, 100 * band_curve.values, where="post", color=color, lw=1.8,
                label=f"{band}: median {km.median_survival(g[col], ~g.live):.0f} days")
    ax.axhline(50, color=INK_2, ls="--", lw=0.8)
    ax.set(title=title, xlabel="days", xlim=(0, 120), ylim=(0, 100))
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel("% still open")
fig.suptitle("Company volume, same listings measured two ways (listings with a posting date)", fontsize=11)
plt.tight_layout()
plt.show()

**What the data shows**

- **The PredictLeads crawler is prompt:** a median 1.5 days from the employer's posting date to first seen (1.1 on hiring platforms, 3.2 on job boards); 34% found within a day.
- **Busy companies are crawled more often** (company sites: a median lag of 9.8 days at 1-4 listings vs. 1.2 at 500+). **That doesn't explain the volume effect on these listings:** for listings with a posting date (41% of the scope), the gap between 1-4 and 500+ listings is 24 vs. 18 days from first seen and 31 vs. 22 days from the posting date (about 1.3 to 1.4 times either way), far smaller than the 44 vs. 12 days across all listings (3.5a).
- **Late-found listings** (7+ days after posting) have long windows from the posting date (47 days), so freshness uses `first_seen` with `posted` alongside.

### 3.4 Pay disclosed

| Check | If the result is… | Then… |
|:--|:--|:--|
| Disclosure rate by US state (needs the state from `location_data`) | Much higher in pay-transparency states (CA, CO, NY, WA) | Add state, or a "pay-law state" flag |
| Disclosure by hourly vs. salaried, and by seniority | Strongly tied to hourly | Its effect may really be job type |
| Days open by disclosure, within company-volume bands | The gap holds | Keep as a factor |
| | The gap disappears | Keep only as the companion flag for pay level |

**As built (2026-10-10):** state comes later (3.7, from `location_data`), so 3.4 uses role level, source, company volume and role bucket. Disclosure is compared within company-volume bands and without likely ghosts (the confounders rule).

In [ ]:
# 3.4a Pay disclosure: who discloses pay, and does disclosing go with a different window? (original listings)
# The raw gap, then within company-volume bands (the confounders rule), and without likely ghosts.
o = jobs.loc[orig.index].assign(pay_disclosed=lambda d: d.salary_hourly_usd.notna())
print("3.4a Share of listings that disclose pay")
display(o.groupby("level").pay_disclosed.mean().to_frame("by role level").style.format("{:.1%}"))

disclosure = o.groupby(["source_type", "volume_band"], observed=True).pay_disclosed.mean().unstack("volume_band").reindex(columns=VOL_BANDS)
print("\n3.4a Share disclosing pay, by company volume and source type")
display(disclosure.style.format("{:.1%}"))

window_by_disclosure = pd.DataFrame({
    "disclosed": km_by(o[o.pay_disclosed], "volume_band"),
    "not disclosed": km_by(o[~o.pay_disclosed], "volume_band"),
    "disclosed, without likely ghosts": km_by(o[o.pay_disclosed & ~o.likely_ghost], "volume_band"),
    "not disclosed, without likely ghosts": km_by(o[~o.pay_disclosed & ~o.likely_ghost], "volume_band"),
}).reindex(VOL_BANDS)
print("\n3.4a Median days open (Kaplan-Meier): pay disclosed vs. not, within company-volume bands")
display(window_by_disclosure.style.format("{:.1f}"))

In [ ]:
# 3.4b How pay is posted, and the hourly pay of each role bucket (listings that disclose pay)
paid = o[o.pay_disclosed]
print("3.4b How pay is posted (time unit, listings that disclose pay)")
display(paid.salary_unit.fillna("(missing)").value_counts(normalize=True).to_frame("share").style.format("{:.1%}"))

pay_by_bucket = (paid.groupby("role_bucket").salary_hourly_usd.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
                 [["count", "10%", "25%", "50%", "75%", "90%"]].sort_values("50%", ascending=False))
print("\n3.4b Hourly pay by role bucket (USD; full-time equivalent for salaried pay)")
display(pay_by_bucket.style.format({"count": "{:,.0f}", "10%": "${:,.2f}", "25%": "${:,.2f}", "50%": "${:,.2f}",
                                    "75%": "${:,.2f}", "90%": "${:,.2f}"}))

In [ ]:
# 3.4-chart Disclosure by company volume and source (left), hourly pay by role bucket (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 1.3]})
disc = 100 * o.groupby(["volume_band", "source_type"], observed=True).pay_disclosed.mean().unstack("source_type").reindex(VOL_BANDS)
for k, (col, color) in enumerate(zip(disc.columns, [BLUE, AQUA, MUTED])):
    ax1.bar(np.arange(len(disc)) + (k - 1) * 0.26, disc[col], width=0.26, color=color, label=col)
ax1.set_xticks(range(len(disc)), disc.index)
ax1.set(title="Share disclosing pay", xlabel="the company's listings so far", ylabel="%")
ax1.legend(fontsize=8, frameon=False, loc="lower center", ncol=3)
ax1.grid(axis="x", visible=False)
order = paid.groupby("role_bucket").salary_hourly_usd.median().sort_values().index
ax2.boxplot([paid.loc[paid.role_bucket == b, "salary_hourly_usd"].dropna() for b in order], vert=False,
            tick_labels=list(order), showfliers=False, patch_artist=True,
            boxprops=dict(facecolor=BLUE, alpha=0.5), medianprops=dict(color=INK))
ax2.set(title="Hourly pay by role bucket (whiskers 1.5x IQR; outliers hidden)", xlabel="USD per hour")
plt.tight_layout()
plt.show()

**What the data shows**

- **Disclosure tracks source more than role level:** hiring platforms disclose pay for 53-64% of listings, company sites 36-52%, job boards 12-16%; by level, 47-56%.
- **Disclosed listings stay open longer at busy companies** (22 vs. 6 days at 500+ listings); much of the "not disclosed" group is listings with no description, which close in days (3.9).
- **Pay by bucket:** software development and IT project management lead (median $72-74 an hour); IT support is lowest ($31), business and data analysis $47.

### 3.5 Reposts and company hiring volume

- **Cross-posted vs. reposted (added 2026-10-10).** EDA_SDA_1 counts every later copy of the same job (same company, title and description) as a repost, so a copy on a second board while the first is still up looks the same as a job relisted after it closed. Split them, using each listing's earlier copies only, so nothing comes from the future:
  - **`is_crosspost`:** an earlier copy is still live when this one appears, from a different source (the platform, or "company site" or "job board" when there's no platform).
  - **`is_relisted`:** earlier copies exist, but none is still live: the job came back after closing.
  - **`n_sources_so_far`:** how many different sources carry the job at the moment this listing appears (1 = only here).
  - **Hypothesis: posting on several boards signals urgency.** Test it, don't assume it: days open (Kaplan–Meier) by `n_sources_so_far` and cross-posted vs. relisted vs. single, within company-volume bands and with and without staffing agencies (2b.1). Applicant-tracking systems often send one posting to several boards automatically, so check within platform too: if the gap disappears there, it's the software, not the employer's urgency.
  - **Copies in different cities aren't cross-posts:** each listing names one location (2b.5), so a job in several cities appears as several listings. The table has no per-listing city yet, so **source stands in for it:** a live earlier copy on a different source is a cross-post; on the same source, most likely another city (`is_same_source_copy`).
  - **Matching (decided after the first run, 2026-10-10):** copies are matched on company and clean title, not the description: boards reformat descriptions, and the strict match found 0 cross-posts (a looser check found 460 same-title copies within 7 days).
- **Reposts** (a decided feature: how strongly should it count?): days open by `times_listed` and by relisted vs. cross-posted; repost rate in the chosen groups; repost rate within company-volume bands, to see whether it adds anything beyond company volume.
- **Company hiring volume:** does high volume mean **more competition, or more openings to win?** Its direction in the ranking depends on the answer. Count each company's listings from before each listing only, so the future isn't used. The companies table has no company size or employee count, so listing volume is our only measure of how much a company hires.
- **For career changers:** do high-volume employers post a larger share of entry-level listings or training wording? If so, volume also signals that a newcomer can get in.

In [ ]:
# 3.5a Company hiring volume, counted from each company's EARLIER listings in any field (1c), the way the model sees it.
# Bands (set in 3.0) count the listing itself: 1-4 / 5-49 / 50-499 / 500+. Raw gap, then within source type (the
# confounders rule), with and without staffing agencies, and without likely ghosts (3.1d).
o = jobs.loc[orig.index]

volume_table = pd.DataFrame({
    "share of listings": o.volume_band.value_counts(normalize=True),
    "all listings": km_by(o, "volume_band"),
    "without staffing agencies": km_by(o[~o.is_staffing], "volume_band"),
    "without likely ghosts": km_by(o[~o.likely_ghost], "volume_band"),
    "career-changer view": km_by(o[o.cc_visible], "volume_band"),
    "entry level": o["level"].eq("entry").groupby(o["volume_band"], observed=True).mean(),
}).reindex(VOL_BANDS)
print("3.5a Median days open (Kaplan-Meier) by company volume (earlier listings only)")
display(volume_table.style.format({"share of listings": "{:.1%}", "all listings": "{:.1f}", "without staffing agencies": "{:.1f}",
                                   "without likely ghosts": "{:.1f}", "career-changer view": "{:.1f}", "entry level": "{:.1%}"}))

print("\n3.5a The same within source type (median days open, Kaplan-Meier)")
display(km_by(o, ["source_type", "volume_band"]).unstack("volume_band").reindex(columns=VOL_BANDS).style.format("{:.1f}"))

In [ ]:
# 3.5-chart Median days open by company volume, all listings vs. without staffing agencies vs. without likely ghosts
vol = pd.DataFrame({"all listings": km_by(o, "volume_band"),
                    "without staffing agencies": km_by(o[~o.is_staffing], "volume_band"),
                    "without likely ghosts": km_by(o[~o.likely_ghost], "volume_band")}).reindex(VOL_BANDS)
fig, ax = plt.subplots(figsize=(9, 3.6))
width = 0.26
for k, (col, color) in enumerate(zip(vol.columns, [BLUE, AQUA, MUTED])):
    xs = np.arange(len(vol)) + (k - 1) * width
    ax.bar(xs, vol[col], width=width, color=color, label=col)
ax.set_xticks(range(len(vol)), [f"{b} listings" for b in vol.index])
ax.set(title="Median days open (Kaplan-Meier) by company volume (earlier listings only)", ylabel="days",
       xlabel="the company's listings so far, this one included")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5b Copies of the same job (same company and clean title): cross-posted, same-source copy, or relisted.
# Matched without the description: boards reformat descriptions, so a strict match found no cross-posts at all.
# Uses only each listing's EARLIER copies and whether one was still live when it appeared (known at that moment).
# Full scale (2026-10-11): running totals within each company + title (sorted by first seen) replace the pairwise
# match, which grows with the square of a group's size (one company's 2,000 "software engineer" listings = 4M pairs).
#   live earlier copy, any source:  the latest last_seen among earlier copies  >= this listing's first_seen
#   live earlier copy, same source: the same, within company + title + source
#   cross-posted: the latest-ending earlier copy is on another source and still live (a lower bound when copies on
#                 both the same and another source are live)
jobs["source"] = jobs.platform.fillna(jobs.source_type)
key = ["company_id", "title_clean"]
order = jobs.sort_values("first_seen").index
k = jobs.loc[order, key + ["source", "first_seen", "last_seen"]].dropna(subset=key)
g_all = k.groupby(key, sort=False)
prev_any = g_all.last_seen.cummax().groupby([k[c] for c in key], sort=False).shift()
g_src = k.groupby(key + ["source"], sort=False)
prev_same = g_src.last_seen.cummax().groupby([k[c] for c in key + ["source"]], sort=False).shift()
earlier = g_all.cumcount()
live_any = prev_any >= k.first_seen
live_same = prev_same >= k.first_seen
live_other = live_any & (prev_any > prev_same.fillna(pd.Timestamp("1970-01-01", tz="UTC")))
jobs["earlier_copies"] = earlier.reindex(jobs.index).fillna(0).astype(int)
jobs["live_same_source"] = live_same.reindex(jobs.index).fillna(False).astype(bool)
jobs["live_other_source"] = live_other.reindex(jobs.index).fillna(False).astype(bool)
del k, g_all, g_src, prev_any, prev_same, earlier, live_any, live_same, live_other, order
gc.collect()
jobs["is_crosspost"] = jobs.live_other_source
jobs["is_same_source_copy"] = ~jobs.is_crosspost & jobs.live_same_source
jobs["is_relisted"] = (jobs.earlier_copies > 0) & ~jobs.is_crosspost & ~jobs.is_same_source_copy
jobs["n_sources_so_far"] = 1 + jobs.is_crosspost.astype(int)  # its own source, plus another carrying a live copy
jobs["copy_type"] = np.select([jobs.is_crosspost, jobs.is_same_source_copy, jobs.is_relisted],
                              ["cross-posted", "same-source copy", "relisted"], default="single")
log("3.5", "cross-posted (another source had the job live)", jobs.is_crosspost.sum())
log("3.5", "relisted (earlier copies, none live)", jobs.is_relisted.sum())

copy_shares = jobs.copy_type.value_counts(normalize=True).rename(index={
    "single": "single (no earlier copy)", "cross-posted": "cross-posted (another source live)",
    "same-source copy": "same-source copy (likely another city)", "relisted": "relisted (earlier copies closed)"})
print("3.5b Copies of the same job")
display(copy_shares.to_frame("share of listings").style.format("{:.1%}"))

In [ ]:
# 3.5c Does posting on several boards signal urgency? Days open by copy type, overall and within source type
# (if the gap disappears there, it's the hiring software, not the employer). Uses every listing (copies included);
# still-open listings are censored.
jobs["sources_band"] = np.where(jobs.n_sources_so_far > 1, "2+", "1")
copy_windows = pd.DataFrame({
    "share of listings": jobs.copy_type.value_counts(normalize=True),
    "median days open": km_by(jobs, "copy_type"),
    "without likely ghosts": km_by(jobs[~jobs.likely_ghost], "copy_type"),
})
print("3.5c Median days open (Kaplan-Meier) by copy type")
display(copy_windows.style.format({"share of listings": "{:.1%}", "median days open": "{:.1f}", "without likely ghosts": "{:.1f}"}))

print("\n3.5c Median days open (Kaplan-Meier) by sources carrying the job, within source type")
display(km_by(jobs, ["source_type", "sources_band"]).unstack("sources_band").style.format("{:.1f}"))

**What the data shows**

- **Company volume is steady and real:** median days open of 44, 34, 28 and 12 across the four bands (earlier listings in any field), the same without staffing agencies and nearly the same without likely ghosts. Entry level is about 10% in every band, so volume is about speed, not access.
- **Copies of the same job:** 70% of listings are single; 21% are same-source copies (most likely the same job in another city); 7.5% are relisted after earlier copies closed; 1.2% are cross-posted.
- **Relisted copies close fast** (median 10.6 days): most likely PredictLeads crawler refreshes, not new hiring. Cross-posts close somewhat faster than single listings (22 vs. 28 days), but not consistently within source type (3.5c), so the urgency idea finds little support.

### 3.6 The scope: the roles TripleTen trains for, by role bucket

**Decided (2026-10-09; applied at the start, in 1d, since 2026-10-10):** the scope is **computer and mathematical listings in the US**, with listings a career changer can't get hidden (3.11). It isn't narrowed to one role: role discovery needs several roles to choose from, and a single role would leave too few listings and too few career-switch examples. Instead, **role bucket** is a dimension inside the scope, defined the same way on both sides, so the listings (what's open) and the career changers in `Profiles.ipynb` (who got there) line up role by role:

| Role bucket | Listings: O*NET codes (2019 SOC) | Profiles: resume titles |
|:--|:--|:--|
| data science | 15-2051 data scientists, 15-2041 statisticians, 15-2031 operations research, 15-1221 research scientists, 15-2099 | data analyst / scientist / engineer, BI, statistician, machine learning |
| database | 15-1242, 15-1243 | database administrator or developer, SQL developer |
| quality assurance | 15-1253 | QA analyst or tester |
| security | 15-1212 | information, cyber or IT security analyst |
| networks and systems | 15-1211, 15-1231, 15-1241, 15-1244 | network or systems administrator, engineer or analyst; DevOps; cloud |
| it support | 15-1232 | help desk, desktop support, IT support |
| software development | 15-1251, 15-1252, 15-1254, 15-1255 | software or web developer, programmer |
| other computer roles | 15-1299 | (none) |
| technical sales (added 2026-10-10) | 41-9031 sales engineers | (none) |
| business and data analysis (added 2026-10-10) | 13-1111 management analysts, 13-1161 market research analysts (titles naming analysis, research or insights; split 2026-10-11) | (none) |
| marketing specialists (split 2026-10-11; since 2026-10-13 a label and an opt-in filter, not hidden by default) | 13-1161 listings whose title names no analysis or research | (none) |
| credentialed math | 15-2011 actuaries, 15-2021 mathematicians | (none) |

The listing side is `role_bucket(onet_code)` (`ONET_DETAILED_BUCKETS` for detailed 15-1299 codes first, then `ONET_ROLE_BUCKETS`, which also covers the older 2010 codes) and the resume side is `RULES["cm_buckets"]`, both in `src/career_switch.py`.

**Steps:**
1. **Map** every computer and mathematical listing to a role bucket from `onet_code`, and report unmapped codes (aggregate counts) so the mapping can be fixed.
2. **Size each bucket** on originals only: listings, listings still open, companies and the top company's share, with and without manager-level listings (3.11; they're hidden per applicant, so the scope keeps them).
3. **Count career changers per bucket** from `Profiles.ipynb`'s modeling table (`career_changer`, by the role at the switch; unique people, after duplicate removal), split by pool: retrained after working, retrained as new grads, and changed roles within tech.
4. **Depth check per bucket:** measure the 95% confidence-interval width of the Kaplan–Meier share still open after 7 days in each sub-group. Enough depth means **±5 points or narrower** (a width of 10 or less), tight enough to show the 5–16 point gaps the EDA found, in at least 80% of sub-groups. Splits are tried from finest to coarsest, combining sub-groups that are too wide: month × source type, then source type, then month, then the whole bucket. The finest split that passes is how finely that bucket is analyzed in section 3. (About 300+ listings per sub-group are needed for ±5, so most buckets won't support all 18 month × source sub-groups.)
5. **Keep a bucket** only if it passes the depth check on the listings side **and** has enough career changers to learn from (at least 30 unique people, counting **all three pools**). The 30 was first set for retrained people only, before the within-tech pool existed; counting retrained only would drop data science (20 retrained, 240 in all), which the compass needs to show is reached through other tech roles. Thin buckets are combined with a close one or analyzed only as part of the scope; their listings always stay in the table. **Credentialed math** (actuaries, mathematicians) is out of the scope from 1d: credential-gated, with no career switches into it. **IT project managers** (O*NET 15-1299.09, about 12% of the scope) have their own bucket since 2026-10-10, on both sides (a Profiles rules change). Record each keep, combine or drop with its reason.
6. **Save the scope** (kept buckets, blockers hidden) as its own table, the base for section 4 and EDA_SDA_3.

**Also described per bucket** (not used to choose buckets): variation in the factors (window, source, company volume, reposts, pay), coverage of key fields (description, salary, `posted`), words per description, and openness to career changers (`entry_share`, `career_open_share`, years asked, degree required).

**Related groups:** the team's source-background fields were set aside in EDA_SDA_1's cut; they live on in its occupation summary (pay, timing, openness) and through O*NET skills and the Career Changers Matrix for role discovery. **The two added buckets** (technical sales, business and data analysis) have no career-switch paths in the frozen profiles, so 3.6e marks them for combined analysis; their listings stay, and the compass shows their 2026 market.

In [ ]:
# 3.6a What's in each role bucket (mapped in 3.0 from the O*NET code; detailed 15-1299 codes first)
other = orig[orig.role_bucket.isin(["other computer roles", "unmapped"])]
print("3.6a What's left in 'other computer roles' and unmapped, by occupation (original listings)")
display(other.groupby(["role_bucket", "occupation"]).size().sort_values(ascending=False).head(15).to_frame("listings")
        .style.format("{:,.0f}"))
print(f"IT project managers (O*NET 15-1299.09): {orig.onet_code.fillna('').str.startswith('15-1299.09').sum():,} original listings")
marketing_research = orig[orig.onet_code.fillna("").str.startswith("13-1161")]
print(f"O*NET 13-1161 split by title: {len(marketing_research):,} original listings; "
      f"{(marketing_research.role_bucket == MARKETING_BUCKET).mean():.0%} labelled marketing specialists (a label and opt-in filter), "
      f"{(marketing_research.role_bucket != MARKETING_BUCKET).mean():.0%} stay in business and data analysis")
print(f"hidden in the app (document management specialists): {orig.hidden_occupation.sum():,} original listings")

In [ ]:
# 3.6b Size each bucket (original listings), with and without manager and leadership listings (hidden per applicant)
def bucket_size(g):
    """One row per role bucket: how big it is and how it behaves."""
    top_share = g.company_id.value_counts(normalize=True).iloc[0]
    return pd.Series({
        "listings": len(g),
        "without manager or leadership": (~g.level.isin(["manager", "leadership"])).sum(),
        "still open": g.live.sum(),
        "companies": g.company_id.nunique(),
        "top company's share": top_share,
        "entry level": (g.level == "entry").mean(),
        "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live),
        "median hourly pay": g.salary_hourly_usd.median(),
    })


sizes = orig.groupby("role_bucket").apply(bucket_size, include_groups=False).sort_values("listings", ascending=False)
print("3.6b Role buckets in the scope (original listings)")
display(sizes.style.format({"listings": "{:,.0f}", "without manager or leadership": "{:,.0f}", "still open": "{:,.0f}",
                            "companies": "{:,.0f}", "top company's share": "{:.1%}", "entry level": "{:.1%}",
                            "median days open (Kaplan-Meier)": "{:.1f}", "median hourly pay": "${:,.2f}"}))

In [ ]:
# 3.6c Career changers per bucket, by pool (Profiles' frozen modeling table; counts only)
profiles = pd.read_pickle(DATA_DIR / "profiles" / "modeling_table.pkl")
changers = profiles[profiles.career_changer]
by_pool = pd.crosstab(changers.target_bucket, changers.pool, margins=True, margins_name="all pools")
print("3.6c Career changers per role bucket, by pool (Profiles)")
display(by_pool.style.format("{:,.0f}"))

In [ ]:
# 3.6-chart Listings vs. career changers per role bucket (the two sides the compass joins)
bucket_view = pd.DataFrame({"listings": sizes["listings"], "career changers": by_pool["all pools"]}).drop(
    index=["all pools", "unmapped"], errors="ignore").fillna(0)
order = bucket_view.sort_values("listings").index
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
ax1.barh(order, bucket_view.loc[order, "listings"], color=BLUE, height=0.65)
ax2.barh(order, bucket_view.loc[order, "career changers"], color=ORANGE, height=0.65)
for ax, col in [(ax1, "listings"), (ax2, "career changers")]:
    for y, v in enumerate(bucket_view.loc[order, col]):
        ax.text(v, y, f" {v:,.0f}", va="center", color=INK_2, fontsize=8.5)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.18)
ax1.set(title="2026 listings (scope, original listings)", xlabel="listings")
ax2.set(title="Career changers who moved into the role (Profiles)", xlabel="people")
plt.tight_layout()
plt.show()

In [ ]:
# 3.6d Depth check: how finely can each bucket be split and still give precise results?
# Measure: Kaplan-Meier share still open after 7 days, with its 95% interval. Precise enough = an interval of
# 10 points or less (+-5), tight enough for the 5-16 point gaps the EDA found. A split is deep enough if at least 80%
# of its sub-groups are that tight. Splits are tried from finest to coarsest, combining sub-groups as the plan says:
# month x source type, then source type only, then month only, then the whole bucket. (Rules fixed before looking.)
from lifelines import KaplanMeierFitter

MAX_WIDTH, MIN_PASS_SHARE = 0.10, 0.80
SPLITS = {"month x source type": ["month", "source_type"], "source type": ["source_type"], "month": ["month"],
          "whole bucket": []}


def ci_width_at_7(g):
    """Width of the 95% interval around the Kaplan-Meier share still open after 7 days (NaN for a single listing)."""
    if len(g) < 2:
        return np.nan
    kmf = KaplanMeierFitter().fit(g.days_open, event_observed=~g.live)
    ci = kmf.confidence_interval_survival_function_
    row = ci[ci.index <= 7].iloc[-1]
    return float(row.iloc[1] - row.iloc[0])


full_months = orig[~orig.partial_month]
depth_rows = {}
for bucket, g in full_months.groupby("role_bucket"):
    found = None
    for name, cols in SPLITS.items():
        widths = g.groupby(cols).apply(ci_width_at_7, include_groups=False) if cols else pd.Series([ci_width_at_7(g)])
        if (widths <= MAX_WIDTH).mean() >= MIN_PASS_SHARE:
            found = (name, len(widths), widths.median())
            break
    depth_rows[bucket] = {"listings (full months)": len(g),
                          "finest deep-enough split": found[0] if found else "none",
                          "sub-groups": found[1] if found else 0,
                          "median interval width": found[2] if found else np.nan}
depth_summary = pd.DataFrame(depth_rows).T
depth_summary["listings side passes"] = depth_summary["finest deep-enough split"] != "none"
print("3.6d Depth check: the finest split where 80% of sub-groups have a 7-day interval of 10 points or less")
display(depth_summary.sort_values("listings (full months)", ascending=False))

In [ ]:
# 3.6e Which buckets get their own analysis (listings are never removed; this only decides how they're analyzed): the listings side must pass the depth check AND the bucket needs
# at least 30 career changers across all three pools (both thresholds fixed before looking at results)
MIN_CHANGERS = 30
decision = pd.DataFrame({
    "listings": sizes["listings"],
    "listings side passes": depth_summary["listings side passes"],
    "career changers": by_pool["all pools"].drop("all pools", errors="ignore"),
})
decision["listings side passes"] = decision["listings side passes"].astype("boolean").fillna(False).astype(bool)
decision["career changers"] = decision["career changers"].fillna(0)
decision["enough career changers"] = decision["career changers"] >= MIN_CHANGERS
decision["verdict"] = np.select(
    [decision["listings side passes"] & decision["enough career changers"],
     decision["listings side passes"] | decision["enough career changers"]],
    ["own analysis and compass paths", "combine with a close bucket"], default="no separate analysis")
print("3.6e Role buckets: own analysis, combine, or none (every listing stays in the in-scope table either way)")
display(decision.sort_values("listings", ascending=False).style.format({"listings": "{:,.0f}", "career changers": "{:,.0f}"}))

**What the data shows**

- **Every bucket is deep enough on the listings side** (7-day intervals of 10 points or less in at least 80% of sub-groups, by source type or finer).
- **Eight buckets get their own analysis and compass paths** (30+ career changers each; quality assurance is the thinnest at 38). Technical sales, business and data analysis, marketing specialists and other computer roles have no career-switch paths in the frozen profiles, so they're analyzed combined; their listings stay.
- **O*NET 13-1161 is mostly marketing:** 76% of its 39K original listings have no analyst, research or insights title. They're labelled marketing specialists (29K; 21% entry level, $28.85 an hour where open to newcomers, 3.11c), leaving business and data analysis at 34K. **Decided 2026-10-13: the title rule will not be hand-checked, so marketing specialists are a label and an opt-in filter, no longer hidden by default;** **IT project management is large** (106K) but mostly reached from another tech role (189 of its 262 career changers).

### 3.7 Blind spots from Part 1

| Blind spot | Check or fix |
|:--|:--|
| The "closed" outcome is wrong for reposted jobs: the earlier listing disappears without being filled | In the analysis (not as a model input), treat a listing that was later reposted as "closed without filling" or censored |
| The 2-day "still open" tolerance is a guess | Distribution of days between `last_seen` and September 30 (the data's end date); a bump near 7 days means it's too tight |
| Converting salaried pay to hourly assumes full time (2,080 hours) | Hourly listings keep their posted rate, so only salaried part-time listings are off; flag them with `contract_types` (part time) |
| Company volume counted within the sample | The companies table has no size or employee count. It does have a description (industry, via NLP in EDA_SDA_3), a ticker (publicly traded), parent and subsidiary companies, and similar companies: check whether these add anything beyond volume |
| State isn't in the table | Add it from `location_data`, if PredictLeads includes it |
| Missing country vs. remote, missing language vs. description | Confirm the EDA_SDA_1 section 10a results |
| `entry_seniority` counts only the `junior` label | Check every seniority label and add any that are entry level, such as intern, to `ENTRY_LEVELS` |

**As built (2026-10-10):** 3.7a–3.7e cover the live tolerance, salaried part-time pay (`pay_part_time_salaried`), the seniority labels and state (`state`, from the location records), with state-level pay. The reposted-closing question is answered by 3.5 (relisted copies close in 10.6 days, read as PredictLeads crawler refreshes); the companies table's extra fields and the no-country check move to 3.8.

In [ ]:
# 3.7a Is the 2-day "still open" tolerance right? Days between each listing's last sighting and September 30.
# A listing counts as live if last seen within 2 days of September 30. If the PL crawler visits weekly, many live listings
# would show gaps of 3-7 days, and a bump near 7 days would mean the tolerance is too tight.
gap = (DATA_END - jobs.last_seen).dt.total_seconds() / 86400
gap_counts = pd.cut(gap, [-np.inf, 1, 2, 3, 5, 7, 10, 14], labels=["0-1", "1-2", "2-3", "3-5", "5-7", "7-10", "10-14"]).value_counts().sort_index()
print("3.7a Days between the last sighting and September 30 (listings last seen in the final 14 days)")
display(gap_counts.rename_axis("days before September 30").to_frame("listings").style.format("{:,.0f}"))
print(f"live under the 2-day rule: {jobs.live.sum():,} | last seen 2-7 days before September 30: {gap.between(2, 7, inclusive='right').sum():,}")

In [ ]:
# 3.7b Salaried part-time pay: the hourly conversion assumes 2,080 hours a year, so a part-time salary reads too low.
# Flagged, kept (hourly listings keep their posted rate, so only salaried part-time listings are affected).
unit = jobs.salary_unit.fillna("")
jobs["pay_part_time_salaried"] = jobs.schedule.eq("part time") & ~unit.str.contains("hour") & jobs.salary_hourly_usd.notna()
log("3.7", "flagged pay_part_time_salaried (part time, salaried: hourly rate understated)", jobs.pay_part_time_salaried.sum())
print(f"part-time listings with salaried pay: {jobs.pay_part_time_salaried.sum():,} of {jobs.salary_hourly_usd.notna().sum():,} with pay")

In [ ]:
# 3.7c Do the seniority labels agree with the title? Title words for each PredictLeads label (share of listings)
title_low = jobs.title.fillna("").str.lower()
words = pd.DataFrame({
    "says senior, lead, principal or staff": title_low.str.contains(r"\b(?:senior|sr\.?|lead|principal|staff)\b", regex=True),
    "says junior, entry, associate or graduate": title_low.str.contains(r"\b(?:junior|jr\.?|entry|associate|graduate|new grad)\b", regex=True),
    "says intern or trainee": title_low.str.contains(r"\bintern(?:ship)?\b|\btrainee\b", regex=True),
    "says manager or director": title_low.str.contains(r"\b(?:manager|director|head of)\b", regex=True),
})
words_by_label = (words.groupby(jobs.seniority.fillna("not_set")).mean()
                  .assign(listings=jobs.seniority.fillna("not_set").value_counts()).sort_values("listings", ascending=False))
print("3.7c Title words by seniority label (share of each label's listings)")
display(words_by_label.style.format({**{c: "{:.1%}" for c in words.columns}, "listings": "{:,.0f}"}))

# Evidence for the senior_title flag: within mid_senior, do senior-titled listings ask more of a newcomer?
mid_senior = jobs.loc[orig.index][lambda d: d.level.eq("mid_senior")]


def reach(g):
    """One column of the comparison: what a group of listings asks of a newcomer."""
    stated = g.years_required.dropna()
    return pd.Series({
        "listings": len(g), "states years": g.years_required.notna().mean(),
        "median years asked (where stated)": stated.median(), "asks 5 years or more (where stated)": (stated >= 5).mean(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "degree required": g.degree_required.astype(float).mean(), "open to career changers (any sign)": g.career_open.astype(float).mean(),
        "median hourly pay": g.salary_hourly_usd.median(), "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live)})


REACH_FORMATS = {"listings": "{:,.0f}", "states years": "{:.1%}", "median years asked (where stated)": "{:.1f}",
                 "asks 5 years or more (where stated)": "{:.1%}", "asks 2 years or fewer (where stated)": "{:.1%}",
                 "degree required": "{:.1%}", "open to career changers (any sign)": "{:.1%}", "median hourly pay": "${:,.2f}",
                 "median days open (Kaplan-Meier)": "{:.1f}"}
senior_vs_rest = pd.DataFrame({"senior title": reach(mid_senior[mid_senior.senior_title]),
                               "other mid_senior": reach(mid_senior[~mid_senior.senior_title])})
print("\n3.7c Within mid_senior: senior-titled listings vs. the rest (original listings)")
display(format_rows(senior_vs_rest, REACH_FORMATS))
del mid_senior

print("\n3.7c Internships (2b.3) by role level: should be mostly entry")
display(jobs[jobs.job_type == "internship"].level.value_counts().to_frame("listings").style.format("{:,.0f}"))

In [ ]:
# 3.7d State, from each listing's location records (loaded for the scope only; only counts and state names are shown).
# For the app's location filter and state-level pay.
first = scope_values(jobs.id, "attributes.location_data",  # the first location record of each in-scope listing
                     fn=lambda v: v[0] if isinstance(v, (list, np.ndarray)) and len(v) and isinstance(v[0], dict) else {})
field_names = pd.Series([k for d in first for k in d]).value_counts()
print("3.7d Fields in the location records (how many listings have each)")
display(field_names.rename_axis("field").to_frame("listings").style.format("{:,.0f}"))
STATE_FIELD = next((f for f in ["state", "region", "state_code"] if f in field_names.index), None)
jobs["state"] = jobs.id.map(first.map(lambda d: d.get(STATE_FIELD) if STATE_FIELD else None)) if STATE_FIELD else None
del first
o = jobs.loc[orig.index]
print(f"state field: {STATE_FIELD} | listings with a state: {o.state.notna().mean():.1%} of the scope")
if STATE_FIELD:
    states = o.state.value_counts()
    state_table = pd.DataFrame({
        "share of listings": (states / len(o)).head(15),
        "median hourly pay": o.groupby("state").salary_hourly_usd.median(),
        "share disclosing pay": o.groupby("state").salary_hourly_usd.apply(lambda s: s.notna().mean()),
        "median days open": km_by(o[o.state.isin(states.head(15).index)], "state"),
    }).dropna(subset=["share of listings"]).sort_values("share of listings", ascending=False)
    print("\n3.7d The 15 states with the most listings: share, median hourly pay and median days open (Kaplan-Meier)")
    display(state_table.style.format({"share of listings": "{:.1%}", "median hourly pay": "${:,.2f}",
                                      "share disclosing pay": "{:.1%}", "median days open": "{:.1f}"}))

In [ ]:
# 3.7-chart Last sightings before September 30 (left), listings and median hourly pay by state (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 1.3]})
ax1.bar(gap_counts.index.astype(str), gap_counts.values, color=[BLUE, BLUE] + [MUTED] * (len(gap_counts) - 2), width=0.6)
ax1.set(title="Days between the last sighting and September 30\n(blue: counted as still open)", xlabel="days", ylabel="listings")
ax1.grid(axis="x", visible=False)
if STATE_FIELD:
    top = states.head(15)
    pay = o.groupby("state").salary_hourly_usd.median().reindex(top.index)
    bar_h(ax2, top, "Listings by state (15 largest), with median hourly pay", "listings")
    for y, state_name in enumerate(top.sort_values().index):  # median hourly pay, at the right edge
        ax2.text(1.02, y, f"${pay[state_name]:.0f}/hr", transform=ax2.get_yaxis_transform(), va="center",
                 color=ORANGE, fontsize=8)
plt.tight_layout()
plt.show()

**What the data shows**

- **The 2-day "still open" tolerance holds:** 225K listings were last seen within 2 days of September 30, then the counts drop (7K at 2-3 days) with no bump near 7 days.
- **Seniority labels mostly agree with titles,** but `mid_senior` mixes levels: 36% of its titles say senior, lead, principal or staff. **Kept whole, with a `senior_title` flag:** senior-titled listings ask more (81% want 5+ years where stated vs. 43%), are rarely open to newcomers (4.9% vs. 17%) and pay more ($77.40 vs. $55.29 an hour). Internships are mostly entry level (25K of 32K).
- **State is known for 90% of listings.** California (13%), Texas (8.8%) and New York (7.5%) lead. **Pay-transparency states disclose pay far more often** (Colorado 68%, Washington 67%, Massachusetts 66%, California 63% vs. Texas 30%, Florida 32%, Georgia 29%).

### 3.8 Joining the companies table

Merge the companies table into the listings (a **left join** on `company_id` = company `id`, so every listing stays). Course standard: one master table from all sources, with keys checked first. Join checks: company ids are unique in the companies table (no row multiplication), and match rate (100% in EDA_SDA_1). Columns to bring in: `ticker` (publicly traded), parent company, subsidiary and similar companies, location, and description (its text goes to EDA_SDA_3). The table has **no company size or employee count**.

**Listings with no country** are dropped in EDA_SDA_1 8b (decided 2026-10-12), so the scope has none.

In [ ]:
# 3.8a Join company facts (left join on company id, so every listing stays; keys checked first).
# Loaded for the scope's companies only. The companies table has no size or employee count.
def first_country(locs):
    """The country of a company's first location record (None if it has none)."""
    return locs[0].get("country") if isinstance(locs, (list, np.ndarray)) and len(locs) and isinstance(locs[0], dict) else None


COMPANY_COLS = ["id", "attributes.ticker", "relationships.parent_company.data.id",
                "relationships.subsidiary_companies.data", "relationships.similar_companies.data", "attributes.location_data"]
company_ids = set(jobs.company_id.dropna())
facts = map_parts("companies", lambda d: d.loc[d["id"].isin(company_ids), d.columns.intersection(COMPANY_COLS)])
dupes = facts["id"].duplicated().sum()
facts = facts.drop_duplicates("id").set_index("id")
company = pd.DataFrame({
    "is_public": facts["attributes.ticker"].notna(),
    "has_parent": facts["relationships.parent_company.data.id"].notna(),
    "n_subsidiaries": facts["relationships.subsidiary_companies.data"].map(lambda v: len(v) if isinstance(v, (list, np.ndarray)) else 0),
    "company_country": facts["attributes.location_data"].map(first_country),
}, index=facts.index)
del facts
rows_before = len(jobs)
jobs = jobs.drop(columns=company.columns, errors="ignore").join(company, on="company_id")
assert len(jobs) == rows_before, "the join changed the number of listings"
for col in ["is_public", "has_parent"]:
    jobs[col] = jobs[col].eq(True)  # unmatched companies: no

print("3.8a Join checks")
display(pd.Series({
    "listings before and after the join": f"{rows_before:,} / {len(jobs):,}",
    "companies in the scope": f"{len(company_ids):,}",
    "matched in the companies table": f"{len(company):,} ({len(company) / len(company_ids):.1%})",
    "duplicate company ids in the companies table": f"{dupes:,}",
    "publicly traded (has a ticker)": f"{company.is_public.mean():.1%} of companies",
    "has a parent company": f"{company.has_parent.mean():.1%} of companies",
    "based in the United States": f"{company.company_country.eq('United States').mean():.1%} of companies",
}).to_frame("value"))

In [ ]:
# 3.8b Do company facts add anything beyond hiring volume? Median days open (Kaplan-Meier) within volume bands
def km_bands(df):
    """Median days open per volume band (empty if no listings)."""
    return km_by(df, "volume_band") if len(df) else pd.Series(dtype=float)


o = jobs.loc[orig.index]
for col, yes, no in [("is_public", "publicly traded", "not publicly traded"), ("has_parent", "has a parent company", "no parent company")]:
    by_fact = pd.DataFrame({
        yes: km_bands(o[o[col]]),
        no: km_bands(o[~o[col]]),
        f"{no}, without staffing agencies": km_bands(o[~o[col] & ~o.is_staffing]),
    }).reindex(VOL_BANDS).assign(**{f"share {yes}": o.groupby("volume_band", observed=True)[col].mean()})
    print(f"3.8b Median days open (Kaplan-Meier) by company volume: {yes} or not")
    display(by_fact.style.format({**{c: "{:.1f}" for c in by_fact.columns[:3]}, by_fact.columns[3]: "{:.1%}"}))

In [ ]:
# 3.8-chart Median days open by company volume for public vs. other companies
fig, ax1 = plt.subplots(figsize=(8, 3.8))
pub = pd.DataFrame({"publicly traded": km_bands(o[o.is_public]), "not publicly traded": km_bands(o[~o.is_public])}).reindex(VOL_BANDS)
for k, (col, color) in enumerate(zip(pub.columns, [BLUE, MUTED])):
    ax1.bar(np.arange(len(pub)) + (k - 0.5) * 0.35, pub[col], width=0.35, color=color, label=col)
ax1.set_xticks(range(len(pub)), pub.index)
ax1.set(title="Median days open (Kaplan-Meier) by company volume", xlabel="the company's listings so far (any field)", ylabel="days")
ax1.legend(fontsize=8, frameon=False)
ax1.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**What the data shows**

- **The companies table matches every in-scope company** (71,911 of 71,919); 5.7% of companies are publicly traded and 12.0% have a parent company.
- **The volume effect is a private-company pattern:** at 500+ listings, public companies' listings stay open a median 22 days vs. 8 for others.
- **Listings with no country** are already dropped in `EDA_SDA_1_Lean` (2a), so the scope has none.

### 3.9 Missing-data patterns

**First classify each field** (structural, censored, or missing for a reason), then treat it. Most of ours aren't truly missing, so very little is imputed.

| Field | Missing (final table) | Why | Handling |
|:--|:--|:--|:--|
| `first_seen`, `last_seen`, title, seniority, occupation group, `onet_code` | ~0% | n/a | Nothing. A listing with no `onet_code` can't get a role bucket, so it's outside the scope (report the count) |
| `last_seen` of **live** listings | 25% still open | **Censored,** not missing | Never imputed or dropped; survival methods handle it |
| `platform` | 61% | **Structural:** company sites have no platform | Not missing; `source_type` says "company site / other" |
| `status` | 38% | Never says "open" | Not used; `last_seen` decides what's live |
| `posted` | 58% | Many sources publish no date | **Not imputed.** Freshness uses `first_seen`; day of week uses the "with `posted`" view; add `has_posted`, and check whether listings with and without it differ |
| Salary | 53% | **Missing for a reason** (pay-transparency laws, hourly vs. salaried) | **Pay level isn't filled in.** `salary_disclosed` stays its own factor. Trees use blanks natively; linear models get a median by role bucket × seniority **plus** a missing flag, fit inside the `Pipeline` on training months only. The app shows "not disclosed" |
| Description and its text flags | 18% | Some sources expose no text | Listing kept; text flags are **unknown** (blank), not False, since EDA_SDA_1's fix; `has_desc` is the flag |
| `years_required` | Blank when no years are stated | **"Not stated" isn't zero** | Kept blank, with a "states years" flag; not stated never counts as a blocker |
| Repost features | Blank without a description | Can't be matched | Kept blank, not 0 (0 would claim "first listing") |
| `tags` | 53% empty | Sparse | Not used; `categories` (99%) instead |
| Company description, location (joined in 3.8) | 10%, 6% | Not in the companies file | "Unknown" category |

**Course standard:** percent missing, missingness **by group** (source, occupation group, role bucket, month, state), and `df.isnull().corr()` to see which fields go missing together. Every imputation or "Unknown" category is justified with plots split by the outcome (closed vs. still open). **Rules:** impute only inside the model `Pipeline`, fit on the training months (leakage rules in `PROJECT.md`, How we keep it honest), and add a missing flag whenever a value is filled in, since missingness itself can carry signal (salary disclosure does).

In [ ]:
# 3.9a Percent missing for every column with any blanks (scope), with each key field's class from the plan above.
# List-valued fields count as missing only when blank, not when empty.
MISSING_CLASS = {
    "posted": "not published by the source: not imputed; `has_posted` flag",
    "salary_hourly_usd": "missing for a reason: not filled in; disclosure is its own factor",
    "platform": "structural: company sites have no platform",
    "status": "not used (`last_seen` decides what's live)",
    "years_required": "not stated (never zero, never a blocker)",
    "state": "not in the location records: \"unknown\" in the app's filter",
    "mentions_remote": "unknown without a description (blank, not False)",
    "days_since_repost_last_listed": "no earlier copy, or no description to match",
    "company_country": "not in the companies table",
    "live_same_source": "only set for copies (3.5)",
    "live_other_source": "only set for copies (3.5)",
}
for c in ["salary_low_usd", "salary_high_usd", "salary_unit"]:
    MISSING_CLASS[c] = MISSING_CLASS["salary_hourly_usd"]
for c in ["times_listed", "days_since_first_listed", "desc_hash", "urgency", "entry_wording", "changer_wording",
          "degree_required", "equivalent_experience", "career_open"]:
    MISSING_CLASS[c] = "from the description: unknown without one (blank, not False or 0)"
MISSING_CLASS["n_subsidiaries"] = "company not in the companies table"

miss = jobs.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False)
missing_table = pd.DataFrame({
    "share missing": miss,
    "class and handling": miss.index.map(lambda c: MISSING_CLASS.get(c, "")),
})
print("3.9a Share missing, every column with blanks (scope)")
display(missing_table.style.format({"share missing": "{:.1%}"}))
jobs["has_posted"] = jobs.posted.notna()
print(f"columns with no blanks: {(jobs.notna().all()).sum()} of {jobs.shape[1]} | listings with no description: {(~jobs.has_desc).mean():.1%}")

In [ ]:
# 3.9b Missingness by group (original listings): does each key field go missing more for some sources, roles, levels or months?
# If missingness depends on the group, a filled-in value would carry the group's pattern, so these stay blank or flagged.
o = jobs.loc[orig.index]
KEY_MISSING = {
    "no posting date": o.posted.isna(),
    "no pay": o.salary_hourly_usd.isna(),
    "no description": ~o.has_desc,
    "no years stated": o.years_required.isna(),
    "no state": o.state.isna(),
}
flags = pd.DataFrame(KEY_MISSING).astype(float)
for by, label in [("source_type", "source"), ("role_bucket", "role bucket"), ("level", "role level"), ("month", "month first seen")]:
    by_group = flags.groupby(o[by]).mean().assign(listings=o[by].value_counts())
    print(f"3.9b Share missing by {label}")
    display(by_group.style.format({**{c: "{:.1%}" for c in flags.columns}, "listings": "{:,.0f}"}))

In [ ]:
# 3.9c Which fields go missing together? Correlation of the missing indicators (df.isnull().corr()), fields 1-99% missing
blank = o.drop(columns=["contract_types"], errors="ignore").isna()  # list fields count as present
blank = blank.loc[:, blank.mean().between(0.01, 0.99)].assign(**{"(no description)": ~o.has_desc})
# Fields derived from the same source go missing together by construction (every text flag with the description, every
# pay column with pay), so identical patterns are merged into one group first; only the pairs left are informative.
groups = {}
for c in blank:
    pattern = blank[c].values.tobytes()
    groups.setdefault(pattern, []).append(c)
merged = pd.DataFrame({(g[0] if len(g) == 1 else f"{g[0]} (+{len(g) - 1} more)"): blank[g[0]] for g in groups.values()})
print("3.9c Fields with identical missing patterns (merged)")
display(pd.DataFrame({"fields": [", ".join(g) for g in groups.values() if len(g) > 1]}))

blank_corr = merged.astype(float).corr()
print("\n3.9c Missing together: correlation of missing indicators (strongest pairs)")
display(blank_corr.where(np.triu(np.ones(blank_corr.shape, bool), 1)).stack().rename("correlation")
        .sort_values(key=abs, ascending=False).head(10).to_frame().style.format("{:.2f}"))

In [ ]:
# 3.9d Does missingness go with the outcome? Share missing among closed vs. still-open listings, and median days open
# (Kaplan-Meier) with and without each field. A gap here justifies a missing flag next to any filled-in value.
outcome_rows = {}
for name, m in KEY_MISSING.items():
    m = m.fillna(False).astype(bool)
    outcome_rows[name] = {
        "missing, closed": m[~o.live].mean(), "missing, still open": m[o.live].mean(),
        "median days open, present": km.median_survival(o.days_open[~m], ~o.live[~m]),
        "median days open, missing": km.median_survival(o.days_open[m], ~o.live[m]),
        "missing, without likely ghosts": m[~o.likely_ghost].mean(),
        "median days open, present (with a description)": km.median_survival(o.days_open[~m & o.has_desc], ~o.live[~m & o.has_desc]),
        "median days open, missing (with a description)": km.median_survival(o.days_open[m & o.has_desc], ~o.live[m & o.has_desc])}
outcome_split = pd.DataFrame(outcome_rows).T
print("3.9d Missingness by outcome (original listings)")
display(outcome_split.style.format({c: ("{:.1%}" if c.startswith("missing") else "{:.1f}") for c in outcome_split.columns}))

# Listings with no description close far faster. Are they behind 3.8's fast-closing high-volume private companies?
no_desc_by_type = pd.DataFrame({
    "publicly traded": (~o.has_desc)[o.is_public].groupby(o.volume_band[o.is_public], observed=True).mean(),
    "not publicly traded": (~o.has_desc)[~o.is_public].groupby(o.volume_band[~o.is_public], observed=True).mean(),
}).reindex(VOL_BANDS)
print("\n3.9d Share with no description, by company volume and company type")
display(no_desc_by_type.style.format("{:.1%}"))

with_desc_by_type = pd.DataFrame({
    "publicly traded": km_bands(o[o.is_public & o.has_desc]),
    "not publicly traded": km_bands(o[~o.is_public & o.has_desc]),
}).reindex(VOL_BANDS)
print("\n3.9d Median days open (Kaplan-Meier) by company volume and company type, listings with a description only")
display(with_desc_by_type.style.format("{:.1f}"))

In [ ]:
# 3.9-chart Share missing by role bucket (left), and which fields go missing together (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.1]})
heatmap(ax1, 100 * flags.groupby(o.role_bucket).mean().reindex([b for b in ROLE_ORDER if b in set(o.role_bucket)]),
        "% missing by role bucket", fmt="{:.0f}", cmap="Blues", vmin=0, vmax=100)
key_corr = flags.corr()
heatmap(ax2, key_corr, "Missing together: correlation of missing indicators", fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1)
plt.tight_layout()
plt.show()

**What the data shows**

- **Most blanks are structural or "not stated":** platform (61%: company sites have none), posting date (58%), years asked (54%), pay (52%), recruiter (99%).
- **No description is the key missing field:** 18% of the scope, rising from 8% in April to 23-25% from June, highest on company sites (28%). These listings close in a median 3 days vs. 30, so missing pay or years looks predictive only through them; `has_desc` stays a standard confounder.
- **Missingness tracks source and month, not role bucket** (10-25% no description across buckets). Nothing is imputed here.

### 3.10 Outliers

Boxplots and IQR rules for days open, words per description, hourly pay, times listed. Course standard: inspect, explain, and keep or remove with a reason. Advanced: robust (median / IQR) summaries, and log scales for the right-skewed measures.

In [ ]:
# 3.10a IQR fences for the four skewed measures (original listings). Days open uses closed listings only, since a
# still-open listing's days are unfinished (censored). Right-skewed measures are also fenced on a log scale, which
# flags far fewer points: a long tail is expected there, not an error.
o = jobs.loc[orig.index]
MEASURES = {
    "days open (closed listings)": o.days_open[~o.live],
    "words per description": o.desc_words[o.has_desc],
    "hourly pay (USD)": o.salary_hourly_usd.dropna(),
    "times listed": jobs.times_listed.dropna(),  # all listings: copies are what this counts
}


def fences(s, k=1.5):
    """The lower and upper outlier fences: k times the IQR below the 25th and above the 75th percentile."""
    q1, q3 = s.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


fence_rows = {}
for name, s in MEASURES.items():
    lo, hi = fences(s)
    log_lo, log_hi = np.expm1(fences(np.log1p(s.clip(lower=0))))
    fence_rows[name] = {"values": len(s), "median": s.median(), "IQR": s.quantile(0.75) - s.quantile(0.25),
                        "lower fence": lo, "upper fence": hi, "share above": (s > hi).mean(), "share below": (s < lo).mean(),
                        "share beyond 3xIQR": ((s > fences(s, 3)[1]) | (s < fences(s, 3)[0])).mean(),
                        "upper fence (log scale)": log_hi, "share above (log scale)": (s > log_hi).mean(),
                        "skew": s.skew()}
iqr_table = pd.DataFrame(fence_rows).T
print("3.10a IQR fences (1.5 x IQR, and on a log scale for the skewed measures)")
display(iqr_table.style.format({
    "values": "{:,.0f}", "median": "{:,.2f}", "IQR": "{:,.2f}", "lower fence": "{:,.2f}", "upper fence": "{:,.2f}",
    "share above": "{:.1%}", "share below": "{:.1%}", "share beyond 3xIQR": "{:.1%}",
    "upper fence (log scale)": "{:,.2f}", "share above (log scale)": "{:.1%}", "skew": "{:.2f}"}))

In [ ]:
# 3.10b What are the outliers? Each measure's upper outliers (1.5 x IQR) against the flags that would explain them.
closed = o[~o.live]
long_open_closed = closed.days_open > fences(closed.days_open)[1]
EXPLAIN = ["posted_old", "is_staffing", "is_public", "has_desc"]
print("3.10b Closed listings open longer than the upper fence vs. the rest")
display(pd.DataFrame({"above the fence": closed[long_open_closed][EXPLAIN].mean(),
                      "the rest": closed[~long_open_closed][EXPLAIN].mean()}).style.format("{:.1%}"))

paid = o[o.salary_hourly_usd.notna()]
lo, hi = fences(paid.salary_hourly_usd)
band = np.select([paid.salary_hourly_usd < lo, paid.salary_hourly_usd > hi], ["below the fence", "above the fence"], "inside")
pay_outliers = pd.DataFrame({
    "listings": pd.Series(band, index=paid.index).value_counts(),
    "pay fixed in EDA_SDA_1_Lean 2d (pay_suspect)": paid.groupby(band)["pay_suspect"].mean(),
    "posted hourly": paid.groupby(band)["salary_unit"].apply(lambda s: s.fillna("").str.contains("hour").mean()),
    "senior title, manager or leadership": (paid.senior_title | paid.level.isin(["manager", "leadership"])).groupby(band).mean(),
    "part time or internship": paid.groupby(band).apply(lambda g: (g.schedule.eq("part time") | g.job_type.eq("internship")).mean(),
                                                        include_groups=False),
})
print("\n3.10b Pay outliers: how pay was posted, the pay fix flag, and role level")
display(pay_outliers.style.format({"listings": "{:,.0f}", **{c: "{:.1%}" for c in pay_outliers.columns[1:]}}))
print(f"hourly pay below the federal minimum wage ($7.25): {(paid.salary_hourly_usd < 7.25).sum():,} | above $250 an hour: "
      f"{(paid.salary_hourly_usd > 250).sum():,}")

described = o[o.has_desc]
length_outliers = pd.DataFrame({"under 50 words": described.desc_words.lt(50).groupby(described.source_type).mean(),
                                "above the upper fence": described.desc_words.gt(fences(described.desc_words)[1]).groupby(described.source_type).mean(),
                                "listings": described.source_type.value_counts()})
print("\n3.10b Description length: very short and very long descriptions by source")
display(length_outliers.style.format({"under 50 words": "{:.1%}", "above the upper fence": "{:.1%}", "listings": "{:,.0f}"}))

listed = jobs[jobs.times_listed.notna()]
times_listed_table = pd.DataFrame({"share": listed.times_listed.clip(upper=5).value_counts(normalize=True).sort_index(),
                                   "staffing agencies": listed.is_staffing.groupby(listed.times_listed.clip(upper=5)).mean()}
                                  ).rename(index={5: "5 or more"})
print("\n3.10b Times listed: share of listings by count, and the share from staffing agencies")
display(times_listed_table.style.format("{:.1%}"))

In [ ]:
# 3.10-chart Boxplots of the four measures (whiskers 1.5 x IQR); log scale for days open and words, which are right-skewed
from matplotlib.ticker import ScalarFormatter

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
LOG = {"days open (closed listings)", "words per description"}
for ax, (name, s) in zip(axes, MEASURES.items()):
    ax.boxplot(s.clip(lower=0.5) if name in LOG else s, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor=BLUE, alpha=0.5), medianprops=dict(color=INK),
               flierprops=dict(marker=".", markersize=2, alpha=0.3, markeredgecolor=MUTED))
    if name in LOG:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set(title=name + (" (log scale)" if name in LOG else ""), xticks=[])
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**What the data shows**

- **The long tails are real, not errors:** 6.2% of closed listings run past the IQR fence (82 days), none past the log-scale fence; descriptions over about 1,650 words are 1.2%.
- **Pay outliers are mostly senior roles:** 1.4% are above $146 an hour, 66% of them senior or above. 237 listings under the federal minimum wage and 253 over $250 an hour stay in the data through section 3; 4b flags them (`pay_implausible`) and blanks their hourly pay.
- **Times listed:** 92% are first listings, 3.1% have been listed 5+ times.

### 3.11 Career-change accessibility, and hiding what a career changer can't get

**Decided (2026-10-09; text rules revised 2026-10-13): listings a career changer can't get are hidden where the rule is reliable.** Seniority, likely ghosts and non-tech occupations hide by default. The text rules are keyword rules that EDA_SDA_3 will not hand-validate (decided 2026-10-13), so they are **labels and opt-in filters** the career changer turns on, never hidden by default:

| Blocker | Example | Hidden in |
|:--|:--|:--|
| Seniority, per applicant | Manager level and above (`manager`, `head`, `director`, `vice_president`, `president`, `c_level`, `partner`, `founder`) for applicants without management or senior experience | The app, per profile; this notebook's table keeps them, labelled |
| Text, opt-in filter | Too many years of experience asked; a required license, clearance or sponsorship | EDA_SDA_3's final table, as labels; the app filters only when the user turns it on |
| Text, opt-in filter, per applicant | A degree or certification this applicant doesn't have ("I have no degree") | The app, on request, per profile |
| Occupation, for everyone | Document management specialists (O*NET 15-1299.03): records work, not a tech path (decided 2026-10-10) | The app; flagged `hidden_occupation` in 3.0 and kept in the table |
| Title rules, opt-in filter | Marketing specialists (13-1161 without an analyst title, 3.0); staffing agency from recruiter titles (2b.1). First-pass keyword rules, not hand-checked (decided 2026-10-13) | Labels in the table; the app filters only when the user turns it on |

Report how many listings each rule hides. Then check:

| Check | If the result is… | Then… |
|:--|:--|:--|
| Accessibility flags by company-volume band and by source | Entry-level listings cluster at high-volume employers or on certain platforms | Account for it, so accessibility isn't company size or source in disguise |
| Pay of listings open to career changers vs. the rest, within the scope | A large pay gap | Show it in the app, and let the pay weight trade off against accessibility |
| Days open by accessibility, within the scope | Reachable listings close faster | Raise their urgency in the ranking |
| Degree required by role bucket | Near-universal in a bucket, with little "or equivalent experience" | That bucket is a poor target |
| Each part of `career_open` separately | One part (for example, entry-level wording) adds little | Use a stricter rule for the "open to career changers" view |

In [ ]:
# 3.11a What each hiding rule hides, in the whole scope and among listings open on September 30 (what the app shows).
# Nothing is removed: each rule is a flag, and the app hides by it. Text rules are previews until EDA_SDA_3 validates them.
MANAGER_UP = ["manager", "leadership"]
rules = pd.DataFrame({
    "likely ghost (everyone)": jobs.likely_ghost,
    "hidden occupation (everyone)": jobs.hidden_occupation,
    "manager level or above (applicants without management experience)": jobs["level"].isin(MANAGER_UP),
    "preview: asks 5+ years (everyone, after EDA_SDA_3)": jobs.years_required.ge(5),
    "preview: degree required, no equivalent (applicants without a degree, after EDA_SDA_3)":
        jobs.degree_required.fillna(False).astype(bool) & ~jobs.equivalent_experience.fillna(False).astype(bool),
})
applied = rules.columns[:3]  # the three rules the app applies by default
hidden_now = rules[applied].any(axis=1)
hidden_all = rules.any(axis=1)
summary = pd.DataFrame({"listings": rules.sum(), "share of scope": rules.mean(),
                        "open on September 30": rules[jobs.live].sum(), "share of open": rules[jobs.live].mean()})
summary.loc["any of the first three (a career changer without management experience)"] = [
    hidden_now.sum(), hidden_now.mean(), hidden_now[jobs.live].sum(), hidden_now[jobs.live].mean()]
summary.loc["any rule, previews included"] = [hidden_all.sum(), hidden_all.mean(), hidden_all[jobs.live].sum(), hidden_all[jobs.live].mean()]
print("3.11a What each rule hides (a listing can match several)")
display(summary.style.format({"listings": "{:,.0f}", "share of scope": "{:.1%}", "open on September 30": "{:,.0f}", "share of open": "{:.1%}"}))

In [ ]:
# 3.11b Is accessibility company size or source in disguise? Share open to career changers by volume band x source,
# and each part of `career_open` separately (original listings with a description, where the text parts are known).
o = jobs.loc[orig.index]
od = o[o.has_desc]
PARTS = {"entry-level seniority": "entry_seniority", "asks 2 years or fewer": "low_experience_ask",
         "entry-level or training wording": "entry_wording", "career-changer wording": "changer_wording"}
open_by_cell = (od.career_open.astype(float).groupby([od.volume_band, od.source_type], observed=True).mean()
                .unstack().reindex(VOL_BANDS))
print("3.11b Share open to career changers (any sign), by company volume and source")
display(open_by_cell.style.format("{:.1%}"))

itpm = od.role_bucket.eq("it project management")
print(f"open to career changers, whole scope: {od.career_open.astype(float).mean():.1%} | without IT project management "
      f"(a later move, 9% open): {od.career_open[~itpm].astype(float).mean():.1%}")

part_flags = pd.DataFrame({k: od[v].fillna(False).astype(bool) for k, v in PARTS.items()})
only_this = part_flags.sum(axis=1).eq(1)  # listings where exactly one sign fires
parts_table = (pd.DataFrame({"share of listings": part_flags.mean(),
                             "only sign (share of its listings)": part_flags[only_this].sum() / part_flags.sum().replace(0, np.nan)})
               .join(part_flags.groupby(od.volume_band, observed=True).mean().T.reindex(columns=VOL_BANDS)))
print("\n3.11b Each part of `career_open`: how often it fires, how often it's the only sign, and by company volume")
display(parts_table.style.format("{:.1%}"))

In [ ]:
# 3.11c Do reachable listings pay less or close faster? Within role bucket (pay) and company-volume band (days open),
# listings with a description only (no-description listings close in about 3 days, 3.9) and without likely ghosts.
od = od[~od.likely_ghost]
is_open = od.career_open.fillna(False).astype(bool)
pay_gap = pd.DataFrame({
    "open to career changers": od[is_open].groupby("role_bucket").salary_hourly_usd.median(),
    "not": od[~is_open].groupby("role_bucket").salary_hourly_usd.median(),
    "share open": is_open.groupby(od.role_bucket).mean(),
}).assign(gap=lambda d: d["open to career changers"] - d["not"])
print("3.11c Median hourly pay by role bucket: open to career changers vs. not")
display(pay_gap.style.format({"open to career changers": "${:,.2f}", "not": "${:,.2f}", "share open": "{:.1%}", "gap": "${:,.2f}"}))

window_gap = pd.DataFrame({
    "open to career changers": km_bands(od[is_open]), "not": km_bands(od[~is_open]),
    "entry level only": km_bands(od[od["level"].eq("entry")]),
}).reindex(VOL_BANDS)
print("\n3.11c Median days open (Kaplan-Meier) by company volume: open to career changers vs. not")
display(window_gap.style.format("{:.1f}"))

In [ ]:
# 3.11d Degree required by role bucket (same view as 3.11c: with a description, without likely ghosts): a bucket where nearly every listing requires a degree with no
# "or equivalent experience" is a poor target for career changers without one.
deg = pd.DataFrame({
    "degree required": od.degree_required.astype(float),
    "or equivalent experience": od.equivalent_experience.astype(float),
    "degree, no equivalent": (od.degree_required.fillna(False).astype(bool) & ~od.equivalent_experience.fillna(False).astype(bool)).astype(float),
    "open to career changers": od.career_open.astype(float),
}).groupby(od.role_bucket).mean().assign(listings=od.role_bucket.value_counts())
print("3.11d Degree requirements and openness by role bucket (listings with a description)")
display(deg.sort_values("degree, no equivalent").style.format({**{c: "{:.1%}" for c in deg.columns[:4]}, "listings": "{:,.0f}"}))

In [ ]:
# 3.11-chart What each rule hides among open listings (left); degree with no equivalent vs. openness by role bucket (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.4), gridspec_kw={"width_ratios": [1.1, 1]})
labels = {"likely ghost (everyone)": "likely ghost", "hidden occupation (everyone)": "hidden occupation",
          "manager level or above (applicants without management experience)": "manager level or above",
          "preview: asks 5+ years (everyone, after EDA_SDA_3)": "asks 5+ years (preview)",
          "preview: degree required, no equivalent (applicants without a degree, after EDA_SDA_3)": "degree, no equivalent (preview)"}
bar_h(ax1, 100 * rules[jobs.live].mean().rename(labels), "Share of open listings each rule hides", "% of open listings", "{:.1f}%")
d = deg.sort_values("degree, no equivalent")
y = np.arange(len(d))
ax2.barh(y - 0.2, 100 * d["degree, no equivalent"], height=0.4, color=ORANGE, label="degree required, no equivalent")
ax2.barh(y + 0.2, 100 * d["open to career changers"], height=0.4, color=BLUE, label="open to career changers")
ax2.set_yticks(y, d.index)
ax2.set(title="By role bucket (listings with a description)", xlabel="% of listings")
ax2.legend(fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
ax2.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**What the data shows**

- **What a career changer without management experience doesn't see: 15% of open listings** (likely ghosts 3.7%, manager level or above 11%, hidden occupations 0.4%: document management; marketing specialists are only labelled, with an opt-in filter). An "asks 5+ years" filter would hide another 32% if a user turned it on (EDA_SDA_3 labels it and does not hide it by default); "degree required, no equivalent" fires on only 2.6%.
- **Openness isn't company size or source in disguise:** 18-29% show any sign in every volume band x source cell. Seniority and years asked do most of the work (each is the only sign for 60% and 77% of its listings).
- **Reachable listings pay $10-35 an hour less in every bucket** (smallest in IT support, largest in other computer roles), and close a little faster at small companies (30 vs. 39 days at 1-4 listings).

### 3.12 The retraining compass: the 2026 side

For each kept role bucket, after blockers are hidden: **open listings on September 30, new listings per week, median hourly pay (where disclosed), median days open (Kaplan–Meier), entry-level share, openness to career changers, remote share, and expected job growth** (a placeholder column until the projections are downloaded) (official projections by occupation code; time frame to be decided). Rows in the same role order as Profiles (`ROLE_ORDER` in `src/career_switch.py`: infrastructure → code → data).

Join it by role bucket to Profiles' three path tables, so every path shows its 2026 openings, pay, closing speed and growth:
- **Prior career → first tech job** and **field trained in → first tech job** (4a.2b): the compass for changing careers.
- **Tech job before → after the move** (4a.3a), with the sideways-or-up split: the compass for moving across or up within tech.

Every path carries its people count, and paths under about 10 people are marked thin. Growth sits next to the entry-level share, since projections cover the whole occupation. Its source and method are in job growth tasks in `ANALYSIS.md` (BLS 2025–2035, employment-weighted by bucket, at most 5 years). Aggregates only; this table feeds the app's Retraining compass and the deck (`DIFFERENTIATORS.md`).

In [ ]:
# 3.12a The 2026 market for each role bucket and TripleTen program (original listings; aggregates only), with timing also
# shown without likely ghosts. New listings per week use full weeks (EDA_SDA_1_Lean 3a).
MARKET_ORDER = ROLE_ORDER + EXTRA_ORDER  # the eight computer and mathematical buckets, then the two added TripleTen ones


def market(g):
    """One row of the compass's 2026 side for a group of original listings."""
    return pd.Series({
        "open on September 30": int(g.live.sum()),
        "new listings per week": g.full_week.sum() / FULL_WEEKS,
        "median hourly pay": g.salary_hourly_usd.median(),
        "median days open (Kaplan-Meier)": km.median_survival(g.days_open, ~g.live),
        "entry level": (g.level == "entry").mean(),
        "open to career changers (any sign)": g.career_open.astype(float).mean(),
        "remote": (g.work_arrangement == "remote").mean(),
        "expected growth": np.nan,  # filled from role_growth.pkl (BLS 2025-2035; job growth tasks in ANALYSIS.md) after PredictLeads consents to the join
    })


MARKET_COLS = ["live", "full_week", "salary_hourly_usd", "days_open", "level", "career_open", "work_arrangement", "cc_visible"]


def market_table(df, by, order):
    """The market rows for each group in `by`, plus timing without likely ghosts and for the career-changer view."""
    t = df[MARKET_COLS].groupby(df[by]).apply(market)
    clean = df[~df.likely_ghost]
    t["open, without likely ghosts"] = clean.groupby(by).live.sum()
    t["median days open, without likely ghosts"] = km_by(clean, by)
    seen = df[df.cc_visible]  # the career-changer view (3.1e)
    t["open, career-changer view"] = seen.groupby(by).live.sum()
    t["median days open, career-changer view"] = km_by(seen, by)
    t["median hourly pay, career-changer view"] = seen.groupby(by).salary_hourly_usd.median()
    return t.reindex([r for r in order if r in t.index])


MARKET_FORMATS = {"open on September 30": "{:,.0f}", "new listings per week": "{:,.0f}", "median hourly pay": "${:,.2f}",
                  "median days open (Kaplan-Meier)": "{:.1f}", "entry level": "{:.1%}", "open to career changers (any sign)": "{:.1%}",
                  "remote": "{:.1%}", "open, without likely ghosts": "{:,.0f}", "median days open, without likely ghosts": "{:.1f}",
                  "open, career-changer view": "{:,.0f}", "median days open, career-changer view": "{:.1f}",
                  "median hourly pay, career-changer view": "${:,.2f}"}

o = jobs.loc[orig.index]  # original listings, with the 3.1d ghost flag
role_market = market_table(o[o.role_bucket.isin(MARKET_ORDER)], "role_bucket", MARKET_ORDER)
program_market = market_table(o[~o.hidden_occupation & (o.role_bucket != MARKETING_BUCKET)], "tripleten_program", PROGRAM_ORDER)
# hidden occupations and marketing specialists (by the title label) aren't part of a program's market: Data Analytics trains analysts
print("3.12a The 2026 market by role bucket (original listings; each column shaded on its own scale)")
shade(role_market.drop(columns="expected growth"), MARKET_FORMATS)
print("\n3.12a The 2026 market by TripleTen program (adjacent tech role = no program trains for it)")
shade(program_market.drop(columns="expected growth"), MARKET_FORMATS, highlight="adjacent tech role")

In [ ]:
# 3.12-chart The compass map: hourly pay vs. openness to career changers per bucket (bubble size = open listings)
fig, ax = plt.subplots(figsize=(9, 5))
x = 100 * role_market["open to career changers (any sign)"]
y = role_market["median hourly pay"]
size = 1500 * role_market["open on September 30"] / role_market["open on September 30"].max()
ax.scatter(x, y, s=size, color=BLUE, alpha=0.6, edgecolor=SURFACE)
for b in role_market.index:
    ax.annotate(b, (x[b], y[b]), xytext=(8, 4), textcoords="offset points", fontsize=8.5, color=INK_2)
ax.margins(x=0.2, y=0.12)
ax.set(title="The 2026 market by role: pay vs. openness to career changers (bubble size = open listings)",
       xlabel="% of listings with any sign of openness to career changers", ylabel="median hourly pay ($)")
plt.tight_layout()
plt.show()

In [ ]:
# 3.12b Join the market to Profiles' paths (from the frozen modeling table; counts only). Paths under 10 people are thin.
THIN = 10
retrained = changers[changers.pool.str.startswith("retrained")]
within = changers[changers.pool.eq("changed roles within tech")]
paths = pd.concat([
    retrained.groupby(["background_area", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"background_area": "from"}).assign(path="prior career"),
    retrained.groupby(["retrain_field", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"retrain_field": "from"}).assign(path="field trained in"),
    within.groupby(["from_bucket", "target_bucket"]).size().rename("people").reset_index()
        .rename(columns={"from_bucket": "from"}).assign(path="tech job before the move"),
], ignore_index=True)
paths["thin (under 10 people)"] = paths.people < THIN
compass_2026 = paths.merge(role_market, left_on="target_bucket", right_index=True, how="left")

PATH_COLS = ["people", "thin (under 10 people)", "open on September 30", "median hourly pay", "median days open (Kaplan-Meier)", "entry level"]
for kind in ["prior career", "field trained in", "tech job before the move"]:
    top = compass_2026[compass_2026.path == kind].sort_values("people", ascending=False).head(8)
    print(f"3.12b Largest paths by {kind}, with the 2026 market of the role they lead to")
    display(top.set_index(["from", "target_bucket"])[PATH_COLS].style.format(
        {"people": "{:,.0f}", "open on September 30": "{:,.0f}", "median hourly pay": "${:,.2f}",
         "median days open (Kaplan-Meier)": "{:.1f}", "entry level": "{:.1%}"}))

share_thin = compass_2026.groupby("path")["thin (under 10 people)"].mean()
print("\n3.12b Share of paths resting on fewer than 10 people")
display(share_thin.to_frame("share thin").style.format("{:.1%}"))

In [ ]:
# 3.12c Save the 2026 side for ML_Fit, which assembles the compass and the Ideas tab (aggregates only).
# Written to data/clean/lean/ with the rest of this notebook's tables, so ML_Fit's inputs stay EDA_SDA_2's until this replaces it.
role_market.to_pickle(LEAN_DIR / "role_market.pkl")
program_market.to_pickle(LEAN_DIR / "program_market.pkl")
compass_2026.to_pickle(LEAN_DIR / "compass_paths_2026.pkl")
print(f"saved role_market.pkl ({len(role_market)} buckets), program_market.pkl ({len(program_market)} programs) and "
      f"compass_paths_2026.pkl ({len(compass_2026):,} paths)")

**What the data shows**

- **The 2026 market by bucket:** software development (66K open, 9.2K new a week, $73.54 an hour) and networks and systems (41K open) are the largest; IT support is the door (44% open to career changers, 22% entry level, $31.10); technical sales is the slowest (median 38 days) but the most remote (29%).
- **By TripleTen program:** AI Software Engineering (63K open) and AI & Machine Learning pay the most (about $74); **Quality Assurance and Data Analytics are the most open to career changers** (27% and 26%), Data Analytics at $51.20 now that marketing specialists are out of its market; adjacent tech roles hold 88K open listings.
- **Most paths are thin:** 78% of prior-career paths and 67% of training paths rest on fewer than 10 people (within-tech moves: 43%), so the compass always shows how many people a path rests on. The largest within-tech move is IT support to networks and systems (279 people).

### 3.13 Feature screening: which candidates go forward

Every candidate feature goes through **the same checks**, so the choice of what EDA_SDA_4, TimeSeries and ML_Rank use rests on evidence, not habit. Ten families of candidates:

| Family | Examples | Built in |
|:--|:--|:--|
| 1. Raw listing fields | seniority, source, job type, work arrangement, AI/ML title, recruiter, pay currency | EDA_SDA_1, 2b |
| 2. The listing's own history | times listed, copies, posting lag, batch first sightings | EDA_SDA_1 6.4, 6.8; 3.5 |
| 3. Company history (earlier listings only) | past closing speed of its closed listings, entry-level share, repost and pay-disclosure habits, roles open, hiring trend | 3.13a |
| 4. The market at first sighting | open listings in the same role bucket (and state) that day, new listings that week, weekday and hour | 3.13a |
| 5. Pay in context | hourly pay, its percentile within role bucket and state, range width, hourly vs. salaried | 3.13a |
| 6. Text (keyword flags for now) | years asked, degree, entry-level and career-changer wording, remote, length | EDA_SDA_1; EDA_SDA_3 replaces them |
| 7. Company facts | publicly traded, parent company, subsidiaries | 3.8 |
| 8. Location | state, a pay-transparency-law state | 3.7, 3.13a |
| 9. O*NET Job Zone | how much preparation a role needs | Waiting for the O*NET download |
| 10. Expected job growth | BLS 2025–2035 by occupation | Waiting for PredictLeads' consent to the join |

**The checks, fixed before looking at results:**
- **Outcome:** closed within 7 days of first sighting, on original listings first seen at least 7 days before September 30 (so every one has a full week of follow-up). **Population: the career-changer view (3.1e) decides; the whole scope and the view without senior titles are shown alongside. Repost features are screened on every listing, reposts included (on originals they can't vary).**
- **Coverage:** share with a value, and its lowest share by source type and by month.
- **Leakage:** is the value known when the listing is first seen? Features built from later information are dropped; `last_processed_at` is audited (3.13b).
- **Signal within the confounders:** each feature is cut into levels (true / false, quintiles, or the 10 most common values, with "missing" as its own level). Within each cell of company volume × source type × has a description, the 7-day close rate's spread across levels (levels with 400+ listings), averaged over cells. Some spread appears by chance, more with more levels, so the same spread is measured with the outcome **shuffled within each cell (the noise floor)**, and the signal is the spread above it. **Under 2 points above the noise floor = no signal beyond the confounders.**
- **Stability:** the same spread in April–June and in July–September, and whether the levels keep their order. **Unstable** if one half's spread is under a third of the other's, or the order flips.
- **A second measure (added 2026-10-11): information within the confounders** (conditional mutual information, as a share of the outcome's uncertainty, above the same shuffled noise floor). The spread looks at the gap between levels; information also weighs how many listings sit in each level, so a big gap in a rare level counts for little. **The bar, fixed before looking:** what a yes/no feature splitting listings in half with a 2-point gap would carry. A feature that passes the spread but not the information bar is kept and watched ("the two measures disagree").
- **Redundancy:** Cramér's V between features' levels, on listings where both have a value (a 200,000-listing sample), so sharing blanks (every text flag is blank without a description) doesn't count. **V of 0.7 or more** with a stronger feature = keep one.

A gradient-boosting importance check comes later, at the start of ML_Rank, as a cross-check, not the decision.

In [ ]:
# 3.13a New candidate features, each from what was known when the listing was first seen (earlier listings only)
t0 = jobs.first_seen
by_time = jobs.sort_values("first_seen")


def cumulative(key, value):
    """Per listing, the sum of `value` over the key's EARLIER listings (sorted by first seen), and how many there were."""
    v = value.loc[by_time.index].astype(float).fillna(0)
    k = by_time[key]
    before_sum = v.groupby(k, sort=False).cumsum() - v
    before_n = by_time.groupby(key, sort=False).cumcount()
    return before_sum.reindex(jobs.index), before_n.reindex(jobs.index)


def count_before(key_cols, times):
    """Per listing, how many listings with the same key were first seen strictly before `times`."""
    key = jobs[key_cols].astype(str).agg("|".join, axis=1) if isinstance(key_cols, list) else jobs[key_cols].astype(str)
    cc = pd.DataFrame({"k": key, "first_seen": jobs.first_seen}).sort_values("first_seen")
    cc["cum"] = cc.groupby("k", sort=False).cumcount() + 1
    q = pd.DataFrame({"k": key, "first_seen": times}).reset_index().sort_values("first_seen")
    r = pd.merge_asof(q, cc, on="first_seen", by="k", direction="backward", allow_exact_matches=False)
    return r.set_index("index").cum.reindex(jobs.index).fillna(0)


def open_at_first_sight(key_cols):
    """Per listing, how many listings with the same key were visible (first seen, not yet gone) when it was first seen."""
    key = jobs[key_cols].astype(str).agg("|".join, axis=1) if isinstance(key_cols, list) else jobs[key_cols].astype(str)
    ev = pd.concat([pd.DataFrame({"k": key, "first_seen": jobs.first_seen, "d": 1}),
                    pd.DataFrame({"k": key, "first_seen": jobs.last_seen, "d": -1})]).sort_values("first_seen")
    ev["open"] = ev.groupby("k", sort=False).d.cumsum()
    q = pd.DataFrame({"k": key, "first_seen": t0}).reset_index().sort_values("first_seen")
    r = pd.merge_asof(q, ev[["k", "first_seen", "open"]], on="first_seen", by="k", direction="backward",
                      allow_exact_matches=False)
    return r.set_index("index").open.reindex(jobs.index).fillna(0).clip(lower=0)


# Family 3: company history within the scope, earlier listings only
for name, flag in [("co_entry_share", jobs.level.eq("entry")), ("co_repost_share", jobs.is_repost),
                   ("co_pay_disclosed_share", jobs.salary_hourly_usd.notna())]:
    s, n = cumulative("company_id", flag)
    jobs[name] = (s / n.where(n > 0)).astype("float32")
closed = jobs.loc[~jobs.live, ["company_id", "last_seen", "days_open"]].sort_values("last_seen")
closed["cum_days"] = closed.days_open.clip(upper=120).groupby(closed.company_id, sort=False).cumsum()
closed["cum_n"] = closed.groupby("company_id", sort=False).cumcount() + 1
q = jobs[["company_id", "first_seen"]].reset_index().sort_values("first_seen")
closed["last_seen"] = closed.last_seen + pd.Timedelta(days=2)  # a closure is known about 2 days after the last sighting
past = pd.merge_asof(q, closed.rename(columns={"last_seen": "first_seen"})[["company_id", "first_seen", "cum_days", "cum_n"]],
                     on="first_seen", by="company_id", direction="backward", allow_exact_matches=False).set_index("index")
jobs["co_past_close_days"] = (past.cum_days / past.cum_n).reindex(jobs.index).astype("float32")  # closed before this one appeared
jobs["co_past_closed_n"] = past.cum_n.reindex(jobs.index).fillna(0).astype("float32")
jobs["co_open_now"] = open_at_first_sight("company_id").astype("float32")
c0 = count_before("company_id", t0)
c30 = count_before("company_id", t0 - pd.Timedelta(days=30))
c60 = count_before("company_id", t0 - pd.Timedelta(days=60))
jobs["co_new_30"] = (c0 - c30).astype("float32")
jobs["co_trend"] = ((c0 - c30 + 1) / (c30 - c60 + 1)).astype("float32")  # last 30 days vs. the 30 before
del closed, q, past, c0, c30, c60

# Family 4: the market at first sighting
jobs["state_or_unknown"] = jobs.state.fillna("unknown")
jobs["mkt_open_bucket"] = open_at_first_sight("role_bucket").astype("float32")
jobs["mkt_open_bucket_state"] = open_at_first_sight(["role_bucket", "state_or_unknown"]).astype("float32")
jobs["mkt_new_week_bucket"] = (count_before("role_bucket", t0) - count_before("role_bucket", t0 - pd.Timedelta(days=7))).astype("float32")
jobs["first_weekday"] = t0.dt.day_name()
jobs["first_hour"] = t0.dt.hour

# Family 5: pay in context
mid = (jobs.salary_low_usd + jobs.salary_high_usd) / 2
jobs["pay_range_width"] = ((jobs.salary_high_usd - jobs.salary_low_usd) / mid.where(mid > 0)).astype("float32")
jobs["pay_pct_bucket_state"] = jobs.groupby(["role_bucket", "state_or_unknown"]).salary_hourly_usd.rank(pct=True).astype("float32")
jobs["pay_posted_hourly"] = jobs.salary_unit.fillna("").str.contains("hour").where(jobs.salary_hourly_usd.notna())
jobs["pay_non_usd"] = jobs.salary_currency.notna() & jobs.salary_currency.ne("USD")

# Family 8: pay-transparency laws requiring a pay range in job ads (in force for most of 2026; public sources)
PAY_LAW_STATES = {"California", "Colorado", "Washington", "New York", "Hawaii", "Maryland", "Illinois", "Minnesota",
                  "Vermont", "Massachusetts", "New Jersey"}
jobs["pay_law_state"] = jobs.state.isin(PAY_LAW_STATES).where(jobs.state.notna())

# Family 1 and 2 additions
jobs["has_recruiter"] = jobs.recruiter_key.notna()
s, n = cumulative("recruiter_key", jobs.recruiter_key.notna())
jobs["recruiter_earlier"] = n.where(jobs.recruiter_key.notna(), 0).astype("float32")  # the recruiter's earlier listings; 0 if none named
jobs["posted_lag_days"] = ((jobs.first_seen - jobs.posted).dt.total_seconds() / 86400).astype("float32")
del by_time, s, n
gc.collect()
print("built 21 new candidate features")

In [ ]:
# 3.13b Leakage audit: does `last_processed` (when PredictLeads last processed the listing) move with `last_seen`?
# If it does, it carries the outcome (when the listing disappeared) and can't be a feature.
lp = pd.read_parquet(SCOPE_PATH, columns=["id", "last_processed"])
assert len(lp) == len(jobs) and (lp.id.to_numpy() == jobs.id.to_numpy()).all(), "row order differs from the in-scope table"
lp.index = jobs.index
gap_days = (lp.last_processed - jobs.last_seen).dt.total_seconds() / 86400
print("3.13b `last_processed` vs. `last_seen` (days between them)")
display(pd.Series({
    "last processed within 1 day of last seen": gap_days.abs().le(1).mean(),
    "last processed within 7 days of last seen": gap_days.abs().le(7).mean(),
    "last processed before first seen": (lp.last_processed < jobs.first_seen).mean(),
    "missing": lp.last_processed.isna().mean(),
}).to_frame("share of listings").style.format("{:.1%}"))
LAST_PROCESSED_LEAKS = gap_days.abs().le(7).mean() > 0.5
print("last_processed tracks last_seen (leaks the outcome):", LAST_PROCESSED_LEAKS)
del lp, gap_days

In [ ]:
# 3.13c Screen every candidate: coverage, leakage, signal within the confounders, stability (rules in 3.13)
CANDIDATES = {  # feature: (family, known at first sighting: "yes" / "no" / "partly", note)
    # 1. raw listing fields
    "level": ("1 raw", "yes", ""), "senior_title": ("1 raw", "yes", ""), "platform": ("1 raw", "yes", ""), "job_type": ("1 raw", "yes", ""),
    "is_full_time": ("1 raw", "yes", ""), "is_contract": ("1 raw", "yes", ""), "is_internship": ("1 raw", "yes", ""),
    "work_arrangement": ("1 raw", "yes", ""), "schedule": ("1 raw", "yes", ""), "ai_ml_title": ("1 raw", "yes", ""),
    "title_remote": ("1 raw", "yes", ""), "n_categories": ("1 raw", "yes", ""), "n_tags": ("1 raw", "yes", ""),
    "role_bucket": ("1 raw", "yes", ""), "tripleten_program": ("1 raw", "yes", ""), "occupation": ("1 raw", "yes", ""),
    "is_staffing": ("1 raw", "yes", ""), "on_employer_domain": ("1 raw", "yes", ""),
    "has_recruiter": ("1 raw", "yes", ""), "agency_recruiter": ("1 raw", "yes", "the recruiter has an agency-style title"), "recruiter_earlier": ("1 raw", "yes", "the recruiter's earlier listings"),
    "pay_non_usd": ("1 raw", "yes", ""),
    "last_processed": ("1 raw", "no" if LAST_PROCESSED_LEAKS else "partly", "audited in 3.13b"),
    # 2. the listing's own history
    "times_listed": ("2 history", "yes", ""), "days_since_first_listed": ("2 history", "yes", ""),
    "days_since_repost_last_listed": ("2 history", "yes", ""), "is_repost": ("2 history", "yes", ""),
    "copy_type": ("2 history", "yes", ""), "earlier_copies": ("2 history", "yes", ""),
    "posted_lag_days": ("2 history", "yes", ""), "has_posted": ("2 history", "yes", ""), "posted_old": ("2 history", "yes", ""),
    "batch_first_seen": ("2 history", "partly", "uses the company's later sightings on the same day"),
    "crawl_dropout": ("2 history", "no", "built from when listings disappeared"),
    # 3. company history (earlier listings only)
    "volume_band": ("3 company history", "yes", "standard confounder"), "co_past_close_days": ("3 company history", "yes", ""),
    "co_past_closed_n": ("3 company history", "yes", ""), "co_entry_share": ("3 company history", "yes", ""),
    "co_repost_share": ("3 company history", "yes", ""), "co_pay_disclosed_share": ("3 company history", "yes", ""),
    "co_open_now": ("3 company history", "yes", ""), "co_new_30": ("3 company history", "yes", ""),
    "co_trend": ("3 company history", "yes", ""),
    # 4. the market at first sighting
    "mkt_open_bucket": ("4 market", "yes", ""), "mkt_open_bucket_state": ("4 market", "yes", ""),
    "mkt_new_week_bucket": ("4 market", "yes", ""), "first_weekday": ("4 market", "yes", "PL crawler schedule"),
    "first_hour": ("4 market", "yes", "PL crawler schedule"), "month": ("4 market", "yes", ""),
    # 5. pay in context
    "salary_hourly_usd": ("5 pay", "yes", ""), "pay_range_width": ("5 pay", "yes", ""),
    "pay_pct_bucket_state": ("5 pay", "partly", "ranked against the whole period: recompute on training months"),
    "pay_posted_hourly": ("5 pay", "yes", ""),
    # 6. text (keyword flags; EDA_SDA_3 replaces them)
    "has_desc": ("6 text", "yes", "standard confounder"), "desc_words": ("6 text", "yes", ""),
    "mentions_remote": ("6 text", "yes", ""), "years_required": ("6 text", "yes", ""), "entry_wording": ("6 text", "yes", ""),
    "changer_wording": ("6 text", "yes", ""), "degree_required": ("6 text", "yes", ""),
    "equivalent_experience": ("6 text", "yes", ""), "career_open": ("6 text", "yes", ""),
    "urgency": ("6 text", "yes", "left to EDA_SDA_3's NLP"),
    # 7. company facts
    "is_public": ("7 company facts", "yes", ""), "has_parent": ("7 company facts", "yes", ""),
    "n_subsidiaries": ("7 company facts", "yes", ""),
    # 8. location
    "state": ("8 location", "yes", ""), "pay_law_state": ("8 location", "yes", ""),
}
CONFOUNDERS = {"volume_band": ["source_type", "has_desc"], "source_type": ["volume_band", "has_desc"],
               "has_desc": ["volume_band", "source_type"]}
CANDIDATES["source_type"] = ("1 raw", "yes", "standard confounder")
MIN_WITHIN, MIN_RAW = 400, 500

FOLLOWED = jobs[~jobs.is_repost & (jobs.first_seen <= DATA_END - pd.Timedelta(days=7))]  # a full week of follow-up


def screen_on(base, features=None):
    """Run every check on one population of original listings; returns (table, levels per feature)."""
    global y7, half, CELL, CELL_ALL, Y_SHUFFLED, LEVELS, BASE
    BASE = base
    y7 = (~base.live & base.days_open.le(7)).astype(float)  # closed within 7 days (a full week of follow-up for every listing)
    half = np.select([base.month.astype(str).isin(["2026-04", "2026-05", "2026-06"]),
                      base.month.astype(str).isin(["2026-07", "2026-08", "2026-09"])], ["Apr–Jun", "Jul–Sep"], "other")
    CELL = {k: base[v[0]].astype(str) + "|" + base[v[1]].astype(str) for k, v in CONFOUNDERS.items()}
    CELL_ALL = base.volume_band.astype(str) + "|" + base.source_type.astype(str) + "|" + base.has_desc.astype(str)

    def levels(s):
        """Cut a feature into levels: true / false, quintiles, or the 10 most common values; "missing" is a level."""
        s = s.loc[BASE.index]
        if pd.api.types.is_bool_dtype(s) or str(s.dtype) == "boolean" or set(s.dropna().unique()) <= {True, False}:
            out = s.map({True: "yes", False: "no"})
        elif pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
            out = pd.Series(np.nan, index=s.index, dtype=object)
            known = s.notna()
            out[known] = pd.qcut(s[known].rank(method="first"), 5, labels=["q1", "q2", "q3", "q4", "q5"]).astype(str)
        else:
            top = s.astype(str).where(s.notna()).value_counts().index[:10]
            out = s.astype(str).where(s.astype(str).isin(top), "other").where(s.notna())
        return out.fillna("missing").astype(str)

    def shuffled_within(cell, seed=0):
        # the outcome shuffled within each confounder cell: what spreads look like when a feature carries nothing
        r = np.random.default_rng(seed)
        out = y7.copy()
        for _, idx in y7.groupby(cell).groups.items():
            out.loc[idx] = r.permutation(y7.loc[idx].to_numpy())
        return out

    Y_SHUFFLED = {k: shuffled_within(v) for k, v in {**CELL, "all": CELL_ALL}.items()}

    def within_spread(lv, cell, mask=None, y=None):
        """The 7-day close rate's spread across a feature's levels within each confounder cell, averaged over cells."""
        d = pd.DataFrame({"l": lv, "y": y7 if y is None else y, "c": cell})
        if mask is not None:
            d = d[mask]
        g = d.groupby(["c", "l"]).y.agg(["mean", "size"])
        g = g[g["size"] >= MIN_WITHIN]
        per = g.groupby(level="c")["mean"].agg(lambda m: m.max() - m.min() if len(m) >= 2 else np.nan)
        w = g.groupby(level="c")["size"].sum()[per.notna()]
        return float((per[w.index] * w).sum() / w.sum()) if len(w) else np.nan

    def cond_info(lv, cell, y=None):
        """Information the levels carry about the outcome within the confounder cells (conditional mutual information),
        as a share of the outcome's uncertainty within the cells. Unlike the spread, it weighs how many listings each
        level holds, so a big gap in a rare level counts for little."""
        d = pd.DataFrame({"l": lv, "y": (y7 if y is None else y).astype(int), "c": cell})
        n = d.groupby(["c", "l", "y"]).size().rename("n").reset_index()
        n_cl = n.groupby(["c", "l"]).n.transform("sum")
        n_cy = n.groupby(["c", "y"]).n.transform("sum")
        n_c = n.groupby("c").n.transform("sum")
        N = n.n.sum()
        cmi = (n.n / N * np.log(n.n * n_c / (n_cl * n_cy))).sum()
        cy = n.groupby(["c", "y"]).n.sum()
        c_tot = cy.groupby(level="c").transform("sum")
        h = -(cy / N * np.log(cy / c_tot)).sum()
        return cmi / h if h > 0 else np.nan

    def raw_rates(lv, mask=None):
        """The 7-day close rate of each level with at least MIN_RAW listings, with no confounder held fixed."""
        d = pd.DataFrame({"l": lv, "y": y7})
        if mask is not None:
            d = d[mask]
        g = d.groupby("l").y.agg(["mean", "size"])
        return g.loc[g["size"] >= MIN_RAW, "mean"]

    rows, LEVELS = {}, {}
    for f, (family, known, note) in CANDIDATES.items():
        if f not in jobs or (features is not None and f not in features):
            continue
        lv = levels(jobs[f])
        LEVELS[f] = lv
        cell = CELL.get(f, CELL_ALL)
        present = jobs.loc[base.index, f].notna()
        cov_by = [present.groupby(base.source_type).mean().min(), present.groupby(base.month.astype(str)).mean().min()]
        raw = raw_rates(lv)
        h1, h2 = (raw_rates(lv, half == h) for h in ["Apr–Jun", "Jul–Sep"])
        common = h1.index.intersection(h2.index)
        rows[f] = {
            "family": family, "known at first sighting": known, "coverage": present.mean(), "lowest coverage (source, month)": min(cov_by),
            "spread, raw (points)": 100 * (raw.max() - raw.min()) if len(raw) > 1 else np.nan,
            "spread within confounders (points)": 100 * within_spread(lv, cell),
            "noise floor (points)": 100 * within_spread(lv, cell, y=Y_SHUFFLED.get(f, Y_SHUFFLED["all"])),
            "information above noise (per 10,000)": 1e4 * (cond_info(lv, cell) - cond_info(lv, cell, Y_SHUFFLED.get(f, Y_SHUFFLED["all"]))),
            "within, Apr–Jun": 100 * within_spread(lv, cell, half == "Apr–Jun"),
            "within, Jul–Sep": 100 * within_spread(lv, cell, half == "Jul–Sep"),
            "level order kept (rank correlation)": h1[common].corr(h2[common], method="spearman") if len(common) > 2 else np.nan,
            "note": note,
        }
    screen = pd.DataFrame(rows).T
    screen["signal above noise (points)"] = pd.to_numeric(screen["spread within confounders (points)"]) - pd.to_numeric(screen["noise floor (points)"])
    for c in ["coverage", "lowest coverage (source, month)", "spread, raw (points)", "spread within confounders (points)", "noise floor (points)",
              "within, Apr–Jun", "within, Jul–Sep", "level order kept (rank correlation)", "information above noise (per 10,000)"]:
        screen[c] = pd.to_numeric(screen[c])
    return screen, LEVELS, y7


# Repost features can only vary when reposts are included, so they're screened on every listing (reposts too)
REPOST_FEATURES = ["times_listed", "days_since_first_listed", "days_since_repost_last_listed", "is_repost"]
WITH_REPOSTS = jobs[jobs.cc_visible & (jobs.first_seen <= DATA_END - pd.Timedelta(days=7))]
screen_reposts, _, _ = screen_on(WITH_REPOSTS, features=REPOST_FEATURES)
screen_all_scope, _, _ = screen_on(FOLLOWED)
screen_realistic, _, _ = screen_on(FOLLOWED[FOLLOWED.cc_visible & ~FOLLOWED.senior_title])
screen, LEVELS, y7 = screen_on(FOLLOWED[FOLLOWED.cc_visible])  # the career-changer view decides
base = FOLLOWED[FOLLOWED.cc_visible]
screen.loc[screen_reposts.index, screen_reposts.columns] = screen_reposts
screen.loc[screen_reposts.index, "note"] = "screened on every listing, reposts included"
screen["signal above noise, whole scope (points)"] = screen_all_scope["signal above noise (points)"]
screen["signal above noise, no senior titles (points)"] = screen_realistic["signal above noise (points)"]


# The second measure's bar, fixed before looking: the information a yes/no feature splitting listings in half, with a
# MIN_SIGNAL-point gap at the overall close rate, would carry
def _h(p):
    """The entropy of a yes/no outcome with probability p."""
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


p0 = y7.mean()
MIN_INFO = 1e4 * (_h(p0) - 0.5 * _h(p0 + 0.01) - 0.5 * _h(p0 - 0.01)) / _h(p0)  # MIN_SIGNAL = 2 points: +-1 around the rate
print(f"bar for the second measure: {MIN_INFO:.1f} per 10,000 of the outcome's uncertainty")
print(f"screened {len(screen)} candidates on {len(base):,} original listings in the career-changer view with a full week of follow-up "
      f"({y7.mean():.1%} closed within 7 days)")

In [ ]:
# 3.13d Redundancy: Cramer's V between the candidates' levels (a 200,000-listing sample; listings where both have a value),
# each one's most similar feature, and every feature it overlaps at V of 0.7 or more (src/feature_screen.py, shared with EDA_SDA_4)
from src import feature_screen as fs

screen = fs.redundancy(screen, LEVELS, base.sample(min(len(base), 200_000), random_state=0).index)

In [ ]:
# 3.13e The verdicts (rules from 3.13, applied in this order), saved for EDA_SDA_4, TimeSeries and ML_Rank
MIN_SIGNAL = fs.MIN_SIGNAL
# the rules in this order (src/feature_screen.py): confounder; not known at first sighting; under 5%; signal under 2 points
# (text held for EDA_SDA_3); V of 0.7 or more with any stronger feature = keep one; unstable or little information = watch
screen["verdict"] = [fs.verdict(f, r, screen["signal above noise (points)"], MIN_INFO, CONFOUNDERS) for f, r in screen.iterrows()]
spread_ok = screen["signal above noise (points)"] >= MIN_SIGNAL
info_ok = screen["information above noise (per 10,000)"] >= MIN_INFO
screen["measures agree"] = np.where(spread_ok == info_ok, "yes", np.where(spread_ok, "spread only", "information only"))
agree = screen[["signal above noise (points)", "information above noise (per 10,000)"]].corr(method="spearman").iloc[0, 1]
print(f"the two measures rank the candidates alike: rank correlation {agree:.2f}; they disagree on "
      f"{(screen['measures agree'] != 'yes').sum()} of {len(screen)} candidates")
screen = screen.sort_values(["family", "signal above noise (points)"], ascending=[True, False])
WAITING = pd.DataFrame({"family": ["9 O*NET Job Zone", "10 job growth"], "verdict": ["waiting for data: the O*NET download",
                        "waiting for data: PredictLeads' consent to the BLS join"]}, index=["job_zone", "expected_growth"])
screen_all = pd.concat([screen, WAITING])
screen_all.to_pickle(LEAN_DIR / "feature_screening.pkl")  # data/clean/lean/, with this notebook's other tables

print("3.13e Verdicts by family (candidates)")
display(pd.crosstab(screen_all.family, screen_all.verdict.str.split(":").str[0]))

SHOWN_COLUMNS = ["family", "known at first sighting", "coverage", "signal above noise (points)", "information above noise (per 10,000)",
                 "measures agree", "signal above noise, whole scope (points)", "signal above noise, no senior titles (points)",
                 "spread within confounders (points)", "noise floor (points)", "spread, raw (points)",
                 "within, Apr–Jun", "within, Jul–Sep", "level order kept (rank correlation)", "most similar feature",
                 "Cramér's V with it", f"overlaps (V ≥ {fs.V_REDUNDANT})", "verdict"]
screening_table = screen_all[SHOWN_COLUMNS]
SCREEN_FORMATS = {"coverage": "{:.1%}", "signal above noise (points)": "{:.1f}", "information above noise (per 10,000)": "{:.1f}",
                  "signal above noise, whole scope (points)": "{:.1f}", "signal above noise, no senior titles (points)": "{:.1f}",
                  "spread within confounders (points)": "{:.1f}", "noise floor (points)": "{:.1f}", "spread, raw (points)": "{:.1f}",
                  "within, Apr–Jun": "{:.1f}", "within, Jul–Sep": "{:.1f}", "level order kept (rank correlation)": "{:.2f}",
                  "Cramér's V with it": "{:.2f}"}
print("\n3.13e The screening table (spreads in percentage points of the 7-day close rate)")
display(screening_table.style.format({c: f for c, f in SCREEN_FORMATS.items() if c in screening_table.columns}))
print("saved feature_screening.pkl")

In [ ]:
# 3.13-chart Signal within the confounders, every candidate (colored by verdict)
plot = screen.dropna(subset=["signal above noise (points)"]).sort_values("signal above noise (points)")
colors = [BLUE if v.startswith("keep") else ORANGE if v.startswith("hold") else MUTED for v in plot.verdict]
fig, ax = plt.subplots(figsize=(9, 0.22 * len(plot) + 1.4))
ax.barh(plot.index, plot["signal above noise (points)"], color=colors, height=0.7)
ax.axvline(MIN_SIGNAL, color=INK_2, ls="--", lw=1)
ax.text(MIN_SIGNAL, len(plot) - 0.5, " 2-point bar", color=INK_2, fontsize=8, va="top")
ax.set(title="Signal: spread of the 7-day close rate across each feature's levels, above the noise floor,\nwithin company volume x source x description "
             "(blue: keep; orange: hold for EDA_SDA_3; grey: drop)", xlabel="percentage points above the noise floor")
ax.tick_params(axis="y", labelsize=7.5)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**What the data shows**

- **68 candidates screened, 48 go forward** (1 recomputed on training months, 8 watched), 9 dropped as redundant with a stronger partner, 9 dropped and 2 held for EDA_SDA_3's text checks; 2 more wait for data. Screened on 691K career-changer listings with a full week of follow-up (28.9% closed within 7 days); the whole scope and the view without senior titles agree within 1-3 points, except two: `n_categories` is stronger on the whole scope (14.5 vs. 10.6 points) and `desc_words` weaker without senior titles (7.7 vs. 13.0), so both are read as depending on the population, not as fixed effects.
- **The strongest signals beyond the confounders:** has a description (37 points), the company's past closing speed (29), company volume (24), the platform (21), its past closings (21) and its open listings now (16). **Company history is the strongest new family:** all nine kept. **The second measure (information) agrees:** rank correlation 0.87, disagreeing on 3 of 68. Two rare text flags (career-changer and entry wording) pass the spread but carry little information, so they're watched.
- **Dropped:** AI/ML title (0.8 points), the recruiter's earlier listings (0.3), the agency-style recruiter title (too rare to measure within the confounders), the repost counts once copy type is known, `crawl_dropout` (not known at first sighting) and `last_processed` (it leaks the outcome, 3.13b). `senior_title` sits right at the bar (2.1), so it labels and ranks rather than hides.

## 4. EDA_SDA_2 final table

Every decision in this notebook, defended with its result from sections 2b and 3, then applied all at once in 4b. The table is saved as its own file (`scope_table.parquet`, in `data/clean/lean/` until this notebook replaces `EDA_SDA_2`), so EDA_SDA_1's table stays unchanged. **Hide, never remove:** no listing is dropped here (the scope cut and the US rule are EDA_SDA_1's); everything is a flag the app hides or labels by.

### 4.0 Decisions

| Decision | Evidence (full scale) | Choice | Status |
|:--|:--|:--|:--|
| **The scope** | The roles TripleTen trains for plus adjacent tech roles (EDA_SDA_1 8.0, 8c): 908K listings | As cut by EDA_SDA_1 | Applied in EDA_SDA_1 |
| **Role buckets** | Every bucket is deep enough on the listings side; eight have 30+ career changers (3.6) | Eight buckets with their own analysis and compass paths; technical sales, business and data analysis, marketing specialists and other computer roles analyzed combined (no career-switch paths); document management flagged `hidden_occupation` | Decided; listings always kept |
| **O*NET 13-1161 split by title** | 76% of its 39K original listings are marketing specialists (no analyst, research or insights title) (3.6) | `role_bucket` = marketing specialists, a label and an opt-in filter since 2026-10-13 (a first-pass title rule, not hand-checked; `hidden_occupation` covers document management only) | **Applied in 3.0; hiding revised 2026-10-13** |
| **IT project management** | 9.1% open to career changers, 2.1% entry level, 44% manager level; 189 of its 262 career changers came from another tech role (3.6, 3.11) | Kept as a later move: not suggested as a first tech job unless the prior career has a real path into it | Decided (app and ML_Fit) |
| **Listings with no country** | 8.0% of cleaned listings, 93% from US-based companies; more often remote (30% vs. 13%) (EDA_SDA_1_Lean 4b) | Dropped in EDA_SDA_1_Lean (decided 2026-10-12) | Applied in EDA_SDA_1_Lean 4b |
| **Likely ghosts** | Closing slows with age but never stops; 19% of open listings are older than 90 days; 7.5K likely ghosts (3.5% of open) (3.1) | Flag `likely_ghost`, hidden in the app; other stale listings labelled "open 90+ days" and ranked lower | Decided; timing also shown without them |
| **Manager level or above** | 11% of open listings (3.11) | Labelled, hidden per applicant | Decided (app) |
| **Text blockers** | "Asks 5+ years" would hide 32% of open listings; "degree required, no equivalent" only 2.6% (3.11) | Previews only | Decided 2026-10-13: labels and opt-in filters, not validated (EDA_SDA_3) |
| **Staffing agencies and recruiters** | 2.0% of listings (753 companies, 33 of them from recruiter titles), closing in 12.6 vs. 24.0 days; recruiters with 5+ listings close fast even at employers (8.1 vs. 24.1 days) (2b.1) | Flag `is_staffing` (the app labels "posted by a recruiter"); `recruiter_key` stays a candidate feature; the title rule (`staffing_by_title`) is a label and opt-in filter, not hand-checked (2026-10-13) | Decided |
| **Contract types and where the work happens** | 55% full time, 9.7% contract, 3.3% internship; work arrangement stated for 54% (2b.3, 2b.6) | Employment flags, `job_type`, `work_arrangement`, `schedule` for the app's filters | Decided |
| **AI/ML titles** | 9.0% of listings, across buckets (25% of data science) (2b.4); no closing-speed signal (0.8 points, 3.13) | `ai_ml_title` as a view for the hackathon app's AI/ML audience, not a ranking feature | Decided |
| **Senior titles in `mid_senior`** | 36% of `mid_senior` titles; 81% ask 5+ years vs. 43%, 4.9% vs. 17% open to newcomers, $77 vs. $55 an hour (3.7c); 2.1 points of signal, right at the bar (3.13) | `mid_senior` kept whole; `senior_title` flag; the app shows them ranked lower with a "senior" label | Decided (app, ML_Rank) |
| **The career-changer view** | 86% of original listings, same timing, slightly lower pay; 55% without senior titles (3.1e) | `cc_visible` decides the feature screening; key tables show it beside the whole scope | Decided |
| **`last_processed`** | 88% within 7 days of the last sighting (3.13b) | Never a feature: it leaks the outcome | Decided |
| **Feature screening** | 68 candidates: 48 go forward, 9 redundant, 9 dropped, 2 held, plus 2 waiting for data; a second measure (information) agrees on 65 of 68 (3.13) | The verdicts are saved (`feature_screening.pkl`) for EDA_SDA_4, TimeSeries and ML_Rank | Decided; a gradient-boosting check in ML_Rank cross-checks it |
| **Pay** | 237 listings under $7.25 and 253 over $250 an hour (0.06% of listings); 1,389 part-time listings with salaried pay (3.7, 3.10) | Flag `pay_implausible` and blank its hourly pay; flag `pay_part_time_salaried` | **Applied in 4b** |
| **Missing values** | Mostly structural or "not stated"; no description for 18% (23-25% from June), closing in 3 days (3.9) | Nothing imputed here; `has_desc` a standard confounder; `has_posted` added | Decided |
| **Outliers** | Real tails (3.10) | Keep; times listed grouped 1 / 2–4 / 5 or more (`times_listed_band`) | **Applied in 4b** |
| **"Possibly hard to fill"** | No more open to newcomers than younger listings (16.5% vs. 20.5%) (3.1) | Dropped as a separate label | **Applied in 4b** |

### 4.0 Prioritization factors, after this notebook

| Factor | Result (full scale) | Choice | Status |
|:--|:--|:--|:--|
| Company hiring volume | 12 vs. 44 days (500+ vs. 1–4, from first seen); on the listings with a posting date the gap is about 1.3–1.4× whether measured from first seen or from the posting date, so crawl lag doesn't explain it; a private-company pattern (public 22 vs. others 8 days at 500+) (3.3, 3.5, 3.8) | Keep (`company_earlier`, `volume_band`) | Kept; EDA_SDA_4 tests volume × company type |
| Source and platform | Interacts with volume (company sites 8 vs. 64 days); platforms 10–71 days; 36.0% on the employer's own domain; job boards stop after May (3.2) | Keep `source_type`, `platform`, `on_employer_domain` | Kept; a standard confounder |
| Freshness | Median PredictLeads crawler lag 1.5 days; late-found listings have long windows from posting (3.3) | Keep `first_seen`, with `posted` and `has_posted` | Kept |
| Pay disclosed | Driven by source and state (pay-transparency states 63–68% vs. about 30%) (3.4, 3.7) | Keep, compared within state and source | Kept; EDA_SDA_4 tests it within state |
| Pay level | Reachable listings pay $10–35 an hour less in every bucket (3.11) | Keep, hourly | Decided; the app shows the trade-off |
| Recruiters | Fast closing, beyond staffing agencies, but only 1.3% of listings (2b.1b); naming a recruiter carries 12.5 points of signal, the recruiter's earlier listings none (3.13) | Keep `has_recruiter`; drop `recruiter_earlier` | Screened |
| Company history (new) | The company's past closing speed is the strongest new signal (29 points); entry-level share, pay-disclosure habit, open roles and hiring trend all clear the bar (3.13) | Keep all nine, from earlier listings only | Screened |
| Copies and reposts | 70% single, 21% same-source copies, 7.5% relisted (close in 10.6 days), 1.2% cross-posted (3.5) | Keep `copy_type` and the repost features | Kept; little urgency signal |
| Company type | Public companies' listings stay open longer at high volume (3.8) | Keep `is_public`, `has_parent` | Kept; a standard confounder |
| State | Known for 90%; pay transparency shows (3.7) | Keep `state` | Kept; the app's location filter and a confounder |
| Career-change accessibility | 20% show any sign; similar in every volume band × source cell; seniority and years asked do most of the work (3.11) | Keep `career_open` and its parts | Kept; the text parts are unvalidated keyword rules in EDA_SDA_3 (labels) |

### 4a–4e Applying the decisions and saving the table

In [ ]:
# 4a Settings from the decision table in 4.0
FINAL = {
    "pay_min_hourly": 7.25,             # federal minimum wage; below it, hourly pay is implausible (3.10)
    "pay_max_hourly": 250,              # above it, hourly pay is implausible (3.10)
    "times_listed_bins": [0, 1, 4, np.inf],
    "drop_columns": ["possibly_hard_to_fill"],  # dropped label (3.1)
}
SCOPE_TABLE = LEAN_DIR / "scope_table.parquet"  # EDA_SDA_3's input once this notebook replaces EDA_SDA_2 (then data/clean/)

In [ ]:
# 4b Apply the decisions one at a time, recording what each removes or flags
final = jobs
funnel = [("in-scope listings from EDA_SDA_1_Lean, after cleaning (2b)", len(final), "")]

pay = final.salary_hourly_usd
implausible = pay.lt(FINAL["pay_min_hourly"]) | pay.gt(FINAL["pay_max_hourly"])
final = final.assign(pay_implausible=implausible, salary_hourly_usd=pay.where(~implausible))
funnel.append(("implausible hourly pay: flagged, pay blanked (3.10)", len(final), f"{implausible.mean():.2%} flagged"))

final = final.assign(times_listed_band=pd.cut(final.times_listed, FINAL["times_listed_bins"], labels=["1", "2-4", "5 or more"]))
funnel.append(("times listed grouped (3.10)", len(final), f"{final.times_listed_band.notna().mean():.0%} with a count"))

final = final.drop(columns=FINAL["drop_columns"], errors="ignore")
for flag, label in [("likely_ghost", "likely ghosts (hidden in the app)"), ("hidden_occupation", "document management specialists (hidden in the app)"),
                    ("stale", "open 90+ days (labelled)"), ("is_staffing", "staffing agencies (labelled)")]:
    funnel.append((f"kept, flagged: {label}", len(final), f"{final[flag].mean():.1%} flagged"))

funnel = pd.DataFrame(funnel, columns=["step", "listings", "flags"]).set_index("step")
funnel.insert(1, "removed", (-funnel.listings.diff()).fillna(0).astype(int))
print("4b Final table: listings after each step")
display(funnel.style.format({"listings": "{:,.0f}", "removed": "{:,.0f}"}))

In [ ]:
# 4c Save EDA_SDA_2's final table for EDA_SDA_3 (Parquet; EDA_SDA_1_Lean's table stays unchanged)
final.to_parquet(SCOPE_TABLE)
print("4c Final in-scope table at a glance")
display(pd.Series({
    "listings": f"{len(final):,}",
    "original listings (not reposts)": f"{(~final.is_repost).sum():,}",
    "companies": f"{final.company_id.nunique():,}",
    "role buckets": final.role_bucket.nunique(),
    "still open on September 30": f"{final.live.mean():.1%}",
    "open and not hidden for everyone (no likely ghosts or hidden occupation)":
        f"{(final.live & ~final.likely_ghost & ~final.hidden_occupation).mean():.1%}",
    "columns": final.shape[1],
}).to_frame("value"))
print(f"saved: {SCOPE_TABLE.name} ({SCOPE_TABLE.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# 4d Coverage: how complete the fields this notebook added are (replaces a preview of rows)
ADDED = ["is_staffing", "job_type", "title_clean", "ai_ml_title", "level", "work_arrangement", "schedule", "role_bucket",
         "volume_band", "likely_ghost", "on_employer_domain", "copy_type", "state", "is_public", "company_country"]
print("4d Share of listings with a value, for the fields added here")
display(final[[c for c in ADDED if c in final]].notna().mean().to_frame("share filled").style.format("{:.1%}"))

In [ ]:
# 4e Feature dictionary: every column, its role and meaning. EDA_SDA_1's columns keep their descriptions from
# src/feature_dictionary.py (one source); the columns added here are described below.
from src.feature_dictionary import FEATURES as PART1_FEATURES
NEW_FEATURES = {
    # column: (role, prioritization factor, description)
    "company_earlier": ("factor", "company hiring volume", "The company's listings first seen before this one, in any field (1d); earlier listings only"),
    "volume_band": ("factor", "company hiring volume", "The company's listings so far in any field, this one included: 1-4 / 5-49 / 50-499 / 500+ (3.0)"),
    "is_full_time": ("category", "filter", "A contract label says full time (2b.3)"),
    "co_past_close_days": ("factor", "company history", "Mean days open (capped at 120) of the company's in-scope listings that closed before this one appeared (3.13a)"),
    "co_past_closed_n": ("factor", "company history", "How many of the company's in-scope listings had closed before this one appeared (3.13a)"),
    "co_entry_share": ("factor", "company history", "Share of the company's earlier in-scope listings that were entry level (3.13a)"),
    "co_repost_share": ("factor", "company history", "Share of the company's earlier in-scope listings that were reposts (3.13a)"),
    "co_pay_disclosed_share": ("factor", "company history", "Share of the company's earlier in-scope listings that disclosed pay (3.13a)"),
    "co_open_now": ("factor", "company history", "The company's in-scope listings visible when this one was first seen (3.13a)"),
    "co_new_30": ("factor", "company history", "The company's in-scope listings first seen in the 30 days before this one (3.13a)"),
    "co_trend": ("factor", "company history", "The company's new in-scope listings in the last 30 days vs. the 30 before (+1 each) (3.13a)"),
    "mkt_open_bucket": ("factor", "market", "In-scope listings in the same role bucket visible when this one was first seen (3.13a)"),
    "mkt_open_bucket_state": ("factor", "market", "The same, within the listing's state (or unknown state) (3.13a)"),
    "mkt_new_week_bucket": ("factor", "market", "New listings in the same role bucket in the 7 days before this one (3.13a)"),
    "first_weekday": ("helper", "market", "Weekday first seen (UTC); follows the PL crawler's schedule (3.13a)"),
    "first_hour": ("helper", "market", "Hour first seen (UTC); follows the PL crawler's schedule (3.13a)"),
    "state_or_unknown": ("helper", "", "State, or 'unknown' (3.13a)"),
    "pay_range_width": ("factor", "pay", "Pay range width relative to its midpoint (3.13a)"),
    "pay_pct_bucket_state": ("factor", "pay", "Hourly pay's percentile within role bucket and state; ranked over the whole period, so recomputed on training months for models (3.13a)"),
    "pay_posted_hourly": ("factor", "pay", "Pay posted as an hourly rate (vs. salaried); blank without pay (3.13a)"),
    "pay_non_usd": ("helper", "pay", "Pay posted in a currency other than USD (3.13a)"),
    "pay_law_state": ("factor", "pay", "The state requires a pay range in job ads (in force for most of 2026); blank without a state (3.13a)"),
    "has_recruiter": ("factor", "recruiters", "A recruiter is named (anonymous key) (3.13a)"),
    "recruiter_earlier": ("factor", "recruiters", "The recruiter's earlier in-scope listings; 0 without a recruiter (3.13a)"),
    "posted_lag_days": ("factor", "freshness", "Days from the employer's posting date to first seen; blank without a posting date (3.13a)"),
    "is_part_time": ("category", "filter", "A contract label says part time (2b.3)"),
    "is_contract": ("category", "filter", "A contract label says contract, freelance or 1099 (2b.3)"),
    "is_temporary": ("category", "filter", "A contract label says temporary or seasonal (2b.3)"),
    "is_internship": ("category", "filter", "A contract label says internship (2b.3)"),
    "ai_ml_title": ("category", "role discovery", "The title names AI, machine learning or data science (2b.4); for AI/ML views across buckets"),
    "work_arrangement": ("category", "filter", "Remote, hybrid, onsite, several or not stated, from contract labels and title (2b.6)"),
    "age_on_sep30": ("helper", "", "Days from first seen to September 30, the data's end date (3.1)"),
    "is_staffing": ("factor", "confounder", "Posted by a staffing agency or recruiter (2b.1); the app labels it"),
    "posted_old": ("helper", "freshness", "The employer's posting date is over a year before first seen (2b.2)"),
    "has_posted": ("helper", "freshness", "Has an employer posting date (3.9)"),
    "job_type": ("category", "filter", "Regular, internship or temporary, from contract types and title words (2b.3)"),
    "title_remote": ("helper", "", "The title says remote or work from home (2b.4)"),
    "title_clean": ("helper", "", "Title without location, remote and other tags; used to match copies (2b.4)"),
    "multi_location": ("category", "", "Listed in more than one location (2b.5)"),
    "level": ("category", "career-change accessibility", "Role level: entry (junior), mid_senior (as PredictLeads labels it), manager or leadership (2b.6)"),
    "senior_title": ("factor", "career-change accessibility", "The title says senior, lead, principal, staff or architect (2b.6); tested in 3.7c and 3.13"),
    "cc_visible": ("scope", "", "Visible to a career changer without management experience: no manager level or above, likely ghost or hidden occupation (3.1e)"),
    "schedule": ("category", "filter", "Full time, part time or not stated (2b.6)"),
    "role_bucket": ("category", "role discovery", "Role bucket from the O*NET code, the same buckets as the resumes; 13-1161 split by title (3.0, 3.6)"),
    "hidden_occupation": ("scope", "", "Document management specialists (15-1299.03): hidden in the app (3.0). Marketing specialists are a role-bucket label and opt-in filter, not hidden"),
    "agency_recruiter": ("factor", "recruiters", "The named recruiter has an agency-style title (account, delivery or client manager, bench sales) (2b.1)"),
    "staffing_by_title": ("scope", "", "Flagged as a staffing agency only by its recruiters' titles, not its name (2b.1)"),
    "stale": ("scope", "still open / apply-by", "Open on September 30 and over 90 days old; labelled \"open 90+ days\" and ranked lower (3.1)"),
    "likely_ghost": ("scope", "confounder", "Stale, from a company with no new listing in 60 days or with a posting date over a year old; hidden in the app (3.1)"),
    "on_employer_domain": ("factor", "source", "The listing's URL is on the company's own domain (3.2)"),
    "source": ("category", "source", "Platform name, or the source type where there's no platform (3.5)"),
    "earlier_copies": ("helper", "copies", "Earlier listings with the same company and clean title (3.5)"),
    "live_other_sources": ("helper", "copies", "Other sources carrying a live copy when this one appeared (3.5)"),
    "live_same_source": ("helper", "copies", "An earlier copy was live on the same source; set for copies only (3.5)"),
    "live_other_source": ("helper", "copies", "An earlier copy was live on another source; set for copies only (3.5)"),
    "is_crosspost": ("factor", "copies", "A copy was live on a different source when it appeared (3.5)"),
    "is_same_source_copy": ("factor", "copies", "A copy was live on the same source, likely another city (3.5)"),
    "is_relisted": ("factor", "copies", "Earlier copies had closed; likely a PL crawler refresh (3.5)"),
    "n_sources_so_far": ("factor", "copies", "Sources carrying the job when it appeared, its own included (3.5)"),
    "copy_type": ("category", "copies", "Original, cross-posted, same-source copy or relisted (3.5)"),
    "sources_band": ("category", "copies", "n_sources_so_far grouped: 1 / 2+ (3.5)"),
    "pay_part_time_salaried": ("helper", "pay", "Part time with salaried pay: the hourly rate is understated (3.7)"),
    "state": ("category", "filter", "State from the listing's location records (3.7); a confounder for pay disclosure"),
    "is_public": ("factor", "confounder", "The company has a stock ticker (3.8)"),
    "has_parent": ("factor", "confounder", "The company has a parent company (3.8)"),
    "n_subsidiaries": ("helper", "", "The company's subsidiaries in the companies table (3.8)"),
    "company_country": ("helper", "", "The company's country, from the companies table (3.8)"),
    "pay_implausible": ("scope", "pay", "Hourly pay under $7.25 or over $250: flagged and blanked; the posted range stays (4b)"),
    "times_listed_band": ("factor", "reposts", "times_listed grouped: 1 / 2-4 / 5 or more (4b)"),
}
FEATURES = {**PART1_FEATURES, **NEW_FEATURES}
dictionary = pd.DataFrame(
    [(col, *FEATURES.get(col, ("?", "", "no description yet")), "EDA_SDA_2" if col in NEW_FEATURES else "EDA_SDA_1",
      final[col].notna().mean()) for col in final.columns],
    columns=["column", "role", "prioritization factor", "description", "added in", "coverage"],
).set_index("column")
ROLE_ORDER_DICT = ["identifier", "time", "target", "factor", "category", "text", "scope", "metadata", "helper"]
dictionary = dictionary.sort_values("role", key=lambda s: s.map({r: i for i, r in enumerate(ROLE_ORDER_DICT)}).fillna(99),
                                    kind="stable")
print("4e Final in-scope table: feature dictionary, by role (coverage = share of listings with a value)")
display(dictionary.style.format({"coverage": "{:.1%}"}))
undocumented = [c for c in final.columns if c not in FEATURES]
if undocumented:
    print("columns without a description:", undocumented)

**What the final table shows**

- **The final in-scope table: 908K listings (849K originals) from 72K companies in 12 role buckets (marketing specialists split out, labelled), 128 columns** (`scope_table.parquet`, 0.26 GB). Nothing was removed; every decision is a flag.
- **23.7% are open on September 30 and visible to everyone** (no likely ghost or hidden occupation). Flags kept: likely ghosts 0.9%, open 90+ days 4.7%, staffing agencies 2.0%, hidden occupations 0.4% (document management specialists; marketing specialists are only labelled), implausible pay 0.06%, senior title 36%.
- **The added fields are complete** except state (90%) and company country (92%).

## 5. Conclusion and next steps

**The scope:** 908K listings (849K originals) from 72K companies in 12 role buckets (marketing specialists split out and labelled), cleaned, flagged and saved for EDA_SDA_3 (`scope_table.parquet`). Nothing was removed; every decision is a flag. The full data confirms the sample's main lesson: **how the data was collected explains many of the biggest gaps.** PredictLeads crawl timing, job boards stopping after May and listings with no description (18%, closing in about 3 days) all look like signals until they're checked, so "has a description", company type and state stay standard confounders.

**What goes forward to modeling:** 48 of 68 candidate features, screened on the career-changer view, with a second measure (information) agreeing on 65 of 68 (3.13). The company's own history is the strongest new signal; AI/ML titles, the recruiter's history and the repost counts add nothing beyond the confounders, and `last_processed` leaks the outcome. `mid_senior` stays whole, with senior titles labelled and ranked lower.

### Answers to the clarifying questions

| Question | Answer |
|:--|:--|
| Which long-open listings are ghosts, and which are still hiring? | Company activity separates them. **7.5K likely ghosts are hidden** (3.5% of open listings); other listings open 90+ days stay visible, labelled and ranked lower, since 30–36% of old open listings still close each month (3.1) |
| Does source matter beyond company volume? | Yes, and the two interact: at 500+ listings, company-site listings close in 8 days vs. 22 on hiring platforms; at 1–4, 64 vs. 47. Platforms range from 10 to 71 days (3.2) |
| How precise is freshness? | A median 1.5 days late; busy companies are crawled more often, but that doesn't explain the volume effect: on listings with a posting date the gap is about 1.3–1.4× whether measured from first seen or the posting date (3.3) |
| Is pay disclosure about state laws or source rather than intent? | Largely: pay-transparency states disclose 63–68% vs. about 30% elsewhere, hiring platforms 53–64% vs. job boards 12–16% (3.4, 3.7) |
| Competition or more openings? Are fast recruiters just staffing agencies? | Listings at busy employers close faster (12 vs. 44 days at 500+ vs. 1–4 listings), mostly at private companies; the data has no applicant counts, so volume reads as "apply sooner". Recruiters close fast even at employers (8.1 vs. 24.1 days), not only staffing agencies (2b.1b, 3.3, 3.5, 3.8) |
| How are values missing? | Tied to source and month, not role bucket; no description rises to 23–25% from June. Nothing is imputed here (3.9) |
| How much is left once blockers are hidden? | 85% of open listings stay visible to a career changer without management experience (marketing specialists, 13-1161 without an analyst title, are only labelled since 2026-10-13); an opt-in "5+ years" filter would hide another 32% if turned on; EDA_SDA_3 labels it and does not hide by default (3.11) |
| Which role buckets are deep enough? | All 12 on the listings side (marketing specialists labelled, not hidden by default since 2026-10-13); eight also have enough career changers for their own compass paths. Technical sales, business and data analysis, marketing specialists and other computer roles are analyzed combined (3.6) |
| Which features carry signal beyond the confounders? | 48 of 68 screened go forward. Strongest: has a description, the company's past closing speed, volume, platform, past closings and open roles. Dropped: AI/ML title, the recruiter's history, repost counts, `crawl_dropout`, `last_processed` (3.13) |

### For the career changer
- **IT support is the clearest door** (44% open, 22% entry level) at the lowest pay ($31 an hour); **Quality Assurance and Data Analytics** are the most open TripleTen programs (27% and 26%; Data Analytics at $51.20).
- **Reachable listings pay $10–35 an hour less in every bucket;** the app shows the trade-off, and the compass shows where people moved next (IT support → networks and systems is the most common move, 279 people).
- **Senior-titled listings are mostly out of reach** (81% ask 5+ years, 4.9% open to newcomers): shown with a "senior" label and ranked lower, not hidden.
- **Most paths are thin,** so the compass always shows how many people a path rests on.

### Next steps
- **EDA_SDA_3 (text):** EDA_SDA_2's two title rules (agency-style recruiter titles, the 13-1161 split) are not hand-checked (dropped 2026-10-13): labels and opt-in filters; clean descriptions from `scope_descriptions.parquet`; report the "5+ years" and degree rules as unvalidated labels and opt-in filters (the hand validation was dropped, decided 2026-10-13), so no text blocker hides listings by default; urgency read in context; map skills to O*NET; add evergreen wording to the ghost check.
- **EDA_SDA_4 (statistics):** volume × company type, pay disclosure within state and source, recruiters, and whether reachable listings close faster, in the adjusted Cox model.
- **Ask PredictLeads:** why job-board listings stop after May, and why descriptions go missing more often from June.

---

### Where this draft stands

**All five sections are written and have run on the real file, reading `EDA_SDA_1_Lean`'s tables from `data/clean/lean/`.** The in-scope table it hands on matches `EDA_SDA_2`'s: 908,291 listings, 71,919 companies and 128 columns. Sections 1 to 4 reproduce `EDA_SDA_2`'s printed figures, with the differences below. Everything this notebook writes goes to `data/clean/lean/` (`scope_table.parquet`, `role_market.pkl`, `program_market.pkl`, `compass_paths_2026.pkl`, `feature_screening.pkl`), not beside `EDA_SDA_2`'s files, so `EDA_SDA_3`, `EDA_SDA_4`, `ML_Fit` and the others keep reading the originals until this replaces it. No shared formatting helper is used; the small display functions (`bar_h`, `heatmap`, `format_rows`, `shade`) are defined in the notebook.

**Where this notebook differs from `EDA_SDA_2`, and why**

- **The company-volume claim (3.3, 4.0, 5).** `EDA_SDA_2` says the volume effect is about 1.5 times "like-for-like" from the posting date, "not the nearly 4 times seen from first seen". That compares different sets of listings: the 4 times (44 vs. 12 days) is for all listings, while the 1.5 times is for the 41% with a posting date. On those same listings the gap is 24 vs. 18 days from first seen and 31 vs. 22 from the posting date (about 1.3 and 1.4 times). So crawl lag doesn't explain the gap; the posting-date listings simply show a smaller one. Other documents that quote "about 1.5 times" need the same correction.
- **Pay outliers (3.10).** The original says the 237 and 253 odd-pay listings were blanked in 3.10. They are only counted there; 4b flags them and blanks their hourly pay.
- **Marketing specialists** are listed with the buckets analyzed combined (they have no career changers in the frozen profiles), and their pay where open to newcomers is $28.85 (the original text said $28.90).
- **The no-country figures** in the decision table now come from `EDA_SDA_1_Lean` 4b; the original's came from an older version of the scope.
- **Stale citations** to `STANDARDS.md` point to `PROJECT.md`, How we keep it honest.

**Before this replaces `EDA_SDA_2`**

1. Run `EDA_SDA_3_Lean` (or the original `EDA_SDA_3`) on `data/clean/lean/scope_table.parquet` and check it matches.
2. Move the outputs to `data/clean/` (`LEAN_DIR` in 1a) and point `ML_Fit` at `role_market.pkl`, `program_market.pkl` and `compass_paths_2026.pkl`.
3. Clear the output of the 2b.1 review cell (it shows company names and recruiter titles; it is tagged `data-rows`) before committing.